# 部署 09 · 本地开发 / 客户端调用 / Langfuse 追踪

上一课（`01_为什么需要部署平台与项目骨架.ipynb`）把 Aegra 项目骨架搭了起来，
这一课接着把「它怎么跑起来 → 别人怎么调它 → 出问题怎么查」这条链路走完：

| 环节 | 要回答的问题 | 本课的代码形态 |
|---|---|---|
| 本地开发 | `uv run aegra dev` 一条命令背后自动做了什么？ | `DEV_STEPS` + `section_1_local_dev()` |
| 课案点名的坑 | 目录里已有旧 `docker-compose.yml` 会怎样？ | `section_2_pitfall()` |
| 生产部署 | Dockerfile 每一行在干什么？ | `DOCKERFILE` + `DOCKERFILE_ANNOTATIONS` |
| 客户端调用 | 部署好的 Agent 怎么调？ | `langgraph_sdk` 的 `threads.create()` + `runs.stream()` |
| 流式结构 | `messages/partial` 的 `data` 为什么是两元素列表？ | `consume_stream()` |
| 累计 vs 差量 | 为什么 `content` 要写 `content[printed:]`？ | `build_fake_chunks()` + `show_events=True` |
| 追踪 · 路径 A | 只改 `.env` 四行怎么接 Langfuse？ | `ENV_LINES` + `section_1_env_only()` |
| 追踪 · 路径 B | `graph.py` 里挂 `CallbackHandler` 粒度细在哪？ | `GRAPH_PY_LANGFUSE` + `CallbackHandler` |
| 容器陷阱 | 容器里的 `localhost` 为什么不是宿主机？ | `section_3_docker_localhost()` |

> **本 notebook 由 `Agent/_py_source/09_aegra_deploy/` 下 3 个脚本合并而成**：
> `03_本地开发与生产部署_jxsd.py`（409 行）、`04_客户端调用_jxsd.py`（419 行）、
> `05_对接Langfuse_jxsd.py`（532 行）。三篇本来各是一个独立脚本，这里按
> 「部署 → 调用 → 观测」串成一条链路：**先把服务讲清楚，再讲客户端怎么调，
> 最后讲怎么看见它内部发生了什么**。

**官方文档**
- Aegra 仓库（Apache 2.0，LangGraph Platform 的自托管替代）：<https://github.com/aegra/aegra>
- LangGraph SDK（客户端那套 threads / runs 接口）：<https://docs.langchain.com/langsmith/sdk>
- Langfuse LangChain 集成（`CallbackHandler`）：<https://langfuse.com/integrations/frameworks/langchain>
- Langfuse 自托管：<https://langfuse.com/self-hosting>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务 + 需模型** —— 本课会**真的调一次大模型**，并把它推到本机跑着的 **Langfuse**（`http://localhost:3001`） |
| 依赖 | 标准库（`shutil` / `socket` / `subprocess` / `unicodedata`）+ `httpx` / `langgraph_sdk` / `langchain` / `langfuse`（venv 已装） |
| 密钥 | `settings.api_key`（调模型用，已配置）+ `settings.langfuse_public_key` / `langfuse_secret_key`（推 trace 用，已配置） |
| 前置服务 | ① 本机 **Langfuse**（`localhost:3001`）—— 第 10 节起要它；② 可选：**Docker Desktop** 与 **Aegra 服务**（`localhost:2026`）—— 只有第 4 / 9 节的「在线路径」才需要 |
| 预计耗时 | 约 40 秒（其中模型调用占大头） |

**三个前置条件，缺了都不会报错，只会走各自的降级路径：**

1. **`aegra` CLI 本机没装**（`Aegra 要求独立的 3.12 环境`，本项目规范禁止往这个 venv 里装）。
   所以第 1~4 节以「**命令怎么用 + 配置长什么样 + 本机探测结果**」为主 ——
   探测不到只打印中文提示，**不抛异常**。
2. **Aegra 服务（2026 端口）本机没起**。第 9 节先用 `httpx` 探一次 `/health`：
   通 → 走真实 `langgraph_sdk` 调用；不通 → **降级到离线演示**（手工构造假 chunk，
   把同一段消费逻辑真跑一遍），照样能看清 `messages/partial` 长什么样。
3. **Langfuse**：本机是**真能连**的（`.env` 里已配好 `LANGFUSE_PUBLIC_KEY/SECRET_KEY/HOST`）。
   第 15 节会走**真实** `langfuse.langchain.CallbackHandler` 路径并把 trace 推上去；
   万一密钥为空，也会自动降级成一个「打印型」handler，同样不抛异常。

> ⚠️ **不落地密钥**：本节任何输出块都**不写密钥明文**。源脚本里那行
> `print(f"... {public_key!r}")` 会原样打印本机的公钥，本 notebook 的「预期输出」
> 里把它打码了 —— 你自己跑的时候看到的是 `.env` 里的真实值。

## 本节地图

三篇源文件其实是同一条链路上的三段：**部署 → 调用 → 观测**。

```mermaid
graph TD
    A["源 03 · aegra dev<br/>四件事 + 旧 compose 的坑"] --> B["源 03 · Dockerfile<br/>生产 aegra up / down"]
    B --> C["源 03 · 本机探测<br/>docker / aegra / 2026 / compose"]
    C --> D["源 04 · 客户端那 30 行<br/>langgraph_sdk threads + runs"]
    D --> E["源 04 · messages/partial<br/>content 是累计全文"]
    E --> F["源 04 · 离线假 chunk<br/>与在线共用同一段逻辑"]
    F --> G["源 05 · 路径 A<br/>.env 四行、零代码"]
    G --> H["源 05 · 路径 B<br/>graph.py 挂 CallbackHandler"]
    H --> I["源 05 · 真调一次模型<br/>回调事件 → Langfuse trace"]
```

裸 JupyterLab 不渲染 mermaid，等价表格如下：

| 节 | 讲什么 | 对应源文件 |
|---|---|---|
| 1 | `uv run aegra dev` 自动完成的四件事 + 健康检查 `/health` `/docs` | `03_本地开发与生产部署_jxsd.py` 第 1 节 |
| 2 | 课案点名的坑：目录里已有旧的 `docker-compose.yml` | 第 2 节 |
| 3 | 生产部署：Dockerfile 逐行注释 + `aegra up / down` | 第 3 节 |
| 4 | 本机实测探测：docker / aegra / 2026 端口 / 那个坑的前置条件 | 第 4 节 |
| 5 | 客户端那 30 行：`threads.create()` + `runs.stream()` | `04_客户端调用_jxsd.py` 第 1 节 |
| 6 | `assistant_id` 从哪来：核对真实 `aegra.json` 的 `graphs` key | 第 2 节 |
| 7 | 核心：`messages/partial` 的数据结构 + 累计/差量 | 第 3 节 |
| 8 | 离线降级：手工构造假 chunk（`FakeChunk` / `FakeClient`） | 第 4 节 |
| 9 | 主流程：探测 `/health` → 在线调用 or 离线演示 | 第 5 节 + 主流程 |
| 10~11 | Langfuse 两条接法：`.env` 四行 / `CallbackHandler` | `05_对接Langfuse_jxsd.py` 第 1~2 节 |
| 12~13 | 容器里的 `localhost` 陷阱 / requirements 那 8 个包 | 第 3~4 节 |
| 14~15 | 降级用的「打印型」handler / 本机配置检查 + 真调一次模型 | 第 5 节 + 主流程 |

**与上下节的衔接**：上一课搭好的 `Agent/09_aegra_deploy/aegra_project/` 是**只读素材**——
本课第 4 / 6 节会去读它的 `docker-compose.yml` 与 `aegra.json`，一个字都不改。

## 0. 环境引导

notebook 的工作目录默认是它自己所在的文件夹，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面 `from config import settings` 必 `ModuleNotFoundError`。

> 这一格顺便给出 `NB_DIR`（notebook 所在目录）与 `WORKDIR`（本课临时目录）。
> 源脚本里的 `Path(__file__).resolve().parent` 在本课改写成 `NB_DIR` ——
> notebook 所在目录就是原来脚本所在目录，所以「读同级的 `aegra_project/`」
> 这个行为**与源脚本完全一致**（只读，不写）。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work
```

`仓库根` 是本机实际路径；`临时目录` 落在 notebook 同级（`tmp_nb_work/`，已被 `.gitignore` 覆盖），
本课生成的 `Dockerfile` 会写进它下面本课专属的子目录，不会污染仓库。

### 前置条件自检

本课是 🔴 课，**每一项前置都有一条降级路径**，所以这一格只做「有没有」的体检：

- `aegra` CLI：本机没装属正常（Aegra 要独立的 3.12 环境），缺了第 1~4 节走「讲解 + 探测」；
- `docker` CLI：只有 `aegra dev` / `aegra up` 才需要，缺了不影响本课其它内容；
- `2026` 端口：空闲 = 本地没起 Aegra，第 9 节会自动切离线演示；
- Langfuse：本机**已经在跑**（下面会看到 HTTP 200），第 15 节走真实追踪路径。

这里同时把本课要用的几个 import 提前做掉：`json`（源 04 用）、`shutil` / `socket`
（源 03 的探测用）、`httpx`（探活）、`from config import settings`（源 05 的配置入口）。

In [ ]:
# ===== 前置条件自检（缺什么只打印中文提示，绝不抛异常）=====
import json
import shutil
import socket

import httpx

from config import settings

print("=" * 78)
print("前置条件自检")
print("=" * 78)

_AEGRA = shutil.which("aegra")
print(f"  aegra CLI      ：{_AEGRA or '未安装（本机正常现象）'}")
if not _AEGRA:
    print("    → 本课不靠它跑：安装命令见第 1 节，缺了只走「讲解 + 探测」路径")

_DOCKER = shutil.which("docker")
print(f"  docker CLI     ：{_DOCKER or '未找到'}")
if not _DOCKER:
    print("    → aegra dev / aegra up 依赖它；装 Docker Desktop 后重开终端")

_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
_sock.settimeout(1.0)
_port_2026 = _sock.connect_ex(("127.0.0.1", 2026)) == 0
_sock.close()
print(f"  127.0.0.1:2026 ：{'已被监听（Aegra 可能已在跑）' if _port_2026 else '空闲（本地没起 Aegra）'}")

print(f"  Langfuse 地址  ：{settings.langfuse_host}")
_keys_ready = bool(settings.langfuse_public_key) and bool(settings.langfuse_secret_key)
print(f"  Langfuse 密钥  ：public={'已配置' if settings.langfuse_public_key else '空'} / "
      f"secret={'已配置' if settings.langfuse_secret_key else '空'}")
try:
    _resp = httpx.get(settings.langfuse_host + "/api/public/health", timeout=3.0)
    print(f"  Langfuse 服务  ：HTTP {_resp.status_code} → {'可用' if _resp.status_code == 200 else '异常'}")
except Exception as _exc:   # noqa: BLE001 —— 探不通就降级，不抛异常
    print(f"  Langfuse 服务  ：连不上（{type(_exc).__name__}）")
    print("    → 第 15 节的追踪会走「打印型 handler」降级路径，不报错")

if _keys_ready:
    print("\n  ✅ 本课将走【真实 Langfuse 追踪】路径：模型回调事件会推到上面这个地址")
else:
    print("\n  ⚪ 本课将走【打印型 handler】降级路径：回调事件只打到屏幕上")

### 预期输出

```text
==============================================================================
前置条件自检
==============================================================================
  aegra CLI      ：未安装（本机正常现象）
    → 本课不靠它跑：安装命令见第 1 节，缺了只走「讲解 + 探测」路径
  docker CLI     ：C:\Program Files\Docker\Docker\resources\bin\docker.EXE
  127.0.0.1:2026 ：空闲（本地没起 Aegra）
  Langfuse 地址  ：http://localhost:3001
  Langfuse 密钥  ：public=已配置 / secret=已配置
  Langfuse 服务  ：HTTP 200 → 可用

  ✅ 本课将走【真实 Langfuse 追踪】路径：模型回调事件会推到上面这个地址
```

注意两件事：

1. `aegra CLI` 一行是**未安装** —— 这是本机（也是这份课案）的预期状态，
   Aegra 要求独立的 Python 3.12 环境，本项目规范明确禁止往这个 venv 里装依赖；
2. `Langfuse 服务：HTTP 200` —— 本课第 10~15 节的追踪是**真能跑通**的，
   不是「讲解完就结束」。

## 第一部分 · 本地开发与生产部署（源 `03_local_dev`）

这一篇回答「怎么把它跑起来」，分两段：**本地 `aegra dev`**（开发）与
**`aegra up`**（生产，Dockerfile + compose 全容器化）。

下面先把它用到的小工具建好，再逐节讲。

### 1.1 三个打印小工具：按**显示宽度**对齐表格

Python 的 `len("中文")` 是 **2**（字符数），但它在终端里占 **4 列**（显示宽度）。
直接用 `len()` 做 `ljust` 对齐，中文表格会越排越歪 —— 所以这里按
`unicodedata.east_asian_width` 算「东亚全角字符算 2 列」。

In [ ]:
# 标准库导入（源 03 文件头那几行，原样保留）
import subprocess
import unicodedata


def _disp_width(text: str) -> int:
    """按东亚字符宽度计算字符串在终端里占的列数"""
    return sum(2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1 for ch in text)


def _pad(text: str, width: int) -> str:
    return text + " " * max(0, width - _disp_width(text))


def print_table(title: str, headers: list, rows: list) -> None:
    widths = [
        max(_disp_width(headers[i]), *(_disp_width(str(r[i])) for r in rows))
        for i in range(len(headers))
    ]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(f"\n【{title}】")
    print(line)
    print("| " + " | ".join(_pad(str(headers[i]), widths[i]) for i in range(len(headers))) + " |")
    print(line)
    for row in rows:
        print("| " + " | ".join(_pad(str(row[i]), widths[i]) for i in range(len(row))) + " |")
    print(line)

### 1.2 中文按显示宽度折行（含「避头点」）

英文用 `textwrap` 就行，中文没有空格，只能按显示宽度折。
这里还多做了一步**避头点**：中文排版里标点不能落在行首，
所以折完之后把行首的 `，。；：、）】！？` 并回上一行末尾 ——
不做这一步就会出现「，超时甚至失败」这种以逗号开头的行。

In [ ]:
# 中文没有空格，textwrap 派不上用场；这里按「显示宽度」折行，
# 并优先在中文标点后断开，避免把「超时甚至失败」劈成两半。
_CJK_BREAK_AFTER = "。，；：、）】！？"


def wrap_cjk(text: str, width: int = 66) -> list:
    """按显示宽度折行，优先在中文标点后断开，返回行列表（不丢字符）"""
    lines, cur = [], ""
    for ch in text:
        cur += ch
        if _disp_width(cur) >= width:
            # 回退到最近的标点处断开（最多回退 20 个字符），找不到就硬断
            cut = max((i + 1 for i, c in enumerate(cur) if c in _CJK_BREAK_AFTER), default=-1)
            if cut >= len(cur) - 20:
                lines.append(cur[:cut])
                cur = cur[cut:]
            else:
                lines.append(cur)
                cur = ""
    if cur:
        lines.append(cur)

    # 避头点：中文排版里标点不能落在行首，把它并回上一行末尾。
    # 不做这一步的话，折行结果会出现「，超时甚至失败」这种以逗号开头的行，
    # 读起来像排版事故 —— 这是中文排版（而不是英文）特有的规则。
    for i in range(1, len(lines)):
        while lines[i] and lines[i][0] in _CJK_BREAK_AFTER:
            lines[i - 1] += lines[i][0]
            lines[i] = lines[i][1:]
    # 过滤掉被搬空的行（避免打印出空行）
    return [ln for ln in lines if ln]

### 1.3 探测外部命令：**绝不能抛异常**

这一节的探测要摸 `docker` / `aegra` 两个命令。命令不存在、超时、返回码非 0
都要变成「一句人话」，而不是把 traceback 甩到学员脸上 —— 探测失败本身就是一种结果。

另外 `errors="replace"` 是为中文 Windows 准备的：docker 可能吐 GBK 字节，
解码失败不该让整个探测炸掉。

In [ ]:
def run_cmd(args: list, timeout: int = 10) -> tuple:
    """执行外部命令，返回 (是否成功, 输出文本)

    探测外部命令**绝不能抛异常**：命令不存在、超时、返回码非 0
    都要变成「一句人话」，而不是 traceback 甩到学员脸上。
    """
    try:
        proc = subprocess.run(
            args,
            capture_output=True,
            text=True,
            timeout=timeout,
            encoding="utf-8",
            # errors="replace"：docker 在中文 Windows 上可能吐 GBK 字节，
            # 解码失败不要抛 UnicodeDecodeError，用替换字符顶过去即可 ——
            # 探测的目的是「能不能用」，不是「输出一个字都不能错」。
            errors="replace",
        )
        # stdout 为空时退回 stderr：docker 的版本信息、报错经常只出现在其中一边
        out = (proc.stdout or proc.stderr or "").strip()
        return proc.returncode == 0, out
    except FileNotFoundError:
        return False, f"命令不存在：{args[0]}"
    except subprocess.TimeoutExpired:
        return False, f"命令超时（{timeout}s）：{' '.join(args)}"
    except Exception as exc:   # noqa: BLE001 —— 探测失败本身就是一种结果
        return False, f"{type(exc).__name__}: {exc}"

## 1. `uv run aegra dev` 一条命令做的四件事

课案原文只有一行命令：

```text
# 在项目根目录执行（Docker 需处于运行状态）
uv run aegra dev
```

但它背后自动完成了四件事。把这张表记住，后面排查「服务没起来」时有奇效 ——
出问题的位置不同，看到的报错完全不同。

In [ ]:
DEV_STEPS = [
    ["① 生成 docker-compose.yml", "按 aegra.json / .env 里的参数写一份 compose 文件",
     "⚠️ 目录里已有同名文件时**直接沿用旧的**，不覆盖 —— 本节第 4 部分专门讲"],
    ["② 拉起 PostgreSQL", "docker compose up postgres -d，镜像 pgvector/pgvector:pg18",
     "开发模式只用数据库，不用 Redis（broker 走内存）"],
    ["③ 执行数据库迁移", "建 Aegra 自己的表（threads / runs / checkpoint 等）",
     "空库第一次跑才会真正建表，之后是幂等的"],
    ["④ 热重载启动服务", "uvicorn 监听 2026 端口，改代码自动生效",
     "所以开发时不用反复 Ctrl+C 重启"],
]

下面这一格把四件事打成表格，并给出「起来之后怎么验证服务正常」：
`curl http://localhost:2026/health` 应返回 `{"status": "healthy"}`；
交互式文档在 `http://localhost:2026/docs`（FastAPI 自带的 Swagger UI，
调试阶段直接在页面上试 `threads` / `runs` 接口，比写客户端代码快得多）。

> ⚠️ **Windows 提示**：PowerShell 里 `curl` 是 `Invoke-WebRequest` 的别名，
> 参数不通用。要么写 `curl.exe`，要么用
> `Invoke-RestMethod http://localhost:2026/health`。

In [ ]:
def section_1_local_dev() -> None:
    print("=" * 78)
    print("1. 本地开发：uv run aegra dev")
    print("=" * 78)
    print("  在项目根目录执行（Docker 需处于运行状态）：")
    print("      uv run aegra dev")
    print()
    print_table("aegra dev 自动完成的四件事", ["步骤", "做什么", "注意"], DEV_STEPS)

    # 验证服务正常 —— 课案原文：
    #     curl http://localhost:2026/health
    #     # → {"status": "healthy"}
    print("\n  验证服务正常：")
    print("      curl http://localhost:2026/health")
    print('      # → {"status": "healthy"}')
    # PowerShell 里 curl 是 Invoke-WebRequest 的别名，参数不兼容，
    # 想用真的 curl 得写 curl.exe；或者干脆用 Invoke-RestMethod。
    print("\n  ⚠️ Windows 提示：PowerShell 里 `curl` 是 Invoke-WebRequest 的别名，")
    print("     参数不通用。要么写 curl.exe，要么用 Invoke-RestMethod http://localhost:2026/health。")
    print("\n  交互式 API 文档：http://localhost:2026/docs")
    print("     （FastAPI 自带的 Swagger UI，可以直接在页面上试 threads / runs 接口，")
    print("       调试阶段比写客户端代码快得多。）")


# 主流程第一行（源 03 的横幅）；notebook 里保留它，方便和源脚本的输出对照
print("Agent 课案 · 部署 ③：本地开发与生产部署（aegra dev / aegra up / Dockerfile）")
section_1_local_dev()

### 预期输出

```text
Agent 课案 · 部署 ③：本地开发与生产部署（aegra dev / aegra up / Dockerfile）
==============================================================================
1. 本地开发：uv run aegra dev
==============================================================================
  在项目根目录执行（Docker 需处于运行状态）：
      uv run aegra dev

【aegra dev 自动完成的四件事】
+---------------------------+------------------------------------------------------------+------------------------------------------------------------------------+
| 步骤                      | 做什么                                                     | 注意                                                                   |
+---------------------------+------------------------------------------------------------+------------------------------------------------------------------------+
| ① 生成 docker-compose.yml | 按 aegra.json / .env 里的参数写一份 compose 文件           | ⚠️ 目录里已有同名文件时**直接沿用旧的**，不覆盖 —— 本节第 4 部分专门讲 |
| ② 拉起 PostgreSQL         | docker compose up postgres -d，镜像 pgvector/pgvector:pg18 | 开发模式只用数据库，不用 Redis（broker 走内存）                        |
| ③ 执行数据库迁移          | 建 Aegra 自己的表（threads / runs / checkpoint 等）        | 空库第一次跑才会真正建表，之后是幂等的                                 |
| ④ 热重载启动服务          | uvicorn 监听 2026 端口，改代码自动生效                     | 所以开发时不用反复 Ctrl+C 重启                                         |
+---------------------------+------------------------------------------------------------+------------------------------------------------------------------------+

  验证服务正常：
      curl http://localhost:2026/health
      # → {"status": "healthy"}

  ⚠️ Windows 提示：PowerShell 里 `curl` 是 Invoke-WebRequest 的别名，
     参数不通用。要么写 curl.exe，要么用 Invoke-RestMethod http://localhost:2026/health。

  交互式 API 文档：http://localhost:2026/docs
     （FastAPI 自带的 Swagger UI，可以直接在页面上试 threads / runs 接口，
       调试阶段比写客户端代码快得多。）
```

**这张表其实是「排查地图」**：`aegra dev` 卡在哪一步，看报错就能定位 ——
卡在 ② 是 Docker 没起 / 镜像拉不下来；卡在 ③ 是数据库连不上（**大概率就是第 2 节那个坑**）；
起来了但 `/health` 不通，才是应用本身的问题。

## 2. 课案点名的坑：目录里已有旧的 `docker-compose.yml`

课案原文：

```text
注意：如果目录里已有旧的 docker-compose.yml（比如之前 langgraph 部署留下的），
`aegra dev` 会直接用它而不是生成新的，先删掉或改名。
```

**为什么这个坑特别难查：**

| 现象 | 你看到的 | 真相 |
|---|---|---|
| 不报错 | `aegra dev` 正常把容器起起来了 | 它用了旧文件，但旧文件也是合法 YAML |
| 库连不上 | 「迁移失败」/「连接被拒绝」 | 旧文件里的账号 / 口令 / 端口是**上一套项目**的 |
| 查配置没问题 | `.env` 和 `aegra.json` 都是对的 | 问题根本不在那儿 |

处理办法两条，任选其一：`del docker-compose.yml`（删掉让 `aegra dev` 重新生成）或
`ren docker-compose.yml old.yml`（改名留档）。

In [ ]:
def section_2_pitfall() -> None:
    print("\n" + "=" * 78)
    print("2. 课案点名的坑：目录里已有旧的 docker-compose.yml")
    print("=" * 78)
    print("  课案原文：")
    print("      如果目录里已有旧的 docker-compose.yml（比如之前 langgraph 部署留下的），")
    print("      aegra dev 会直接用它而不是生成新的，先删掉或改名。")
    print("\n  为什么难查：aegra dev 不报错，用旧文件照样把容器起起来；")
    print("  但旧文件里的账号/口令/端口是上一套项目的 → 服务连不上库 →")
    print("  你看到的是「迁移失败 / 连接被拒绝」，而 .env 和 aegra.json 看起来都没问题。")
    print("\n  处理办法（课案给的两条任选其一）：")
    print("      del docker-compose.yml            # 删掉，让 aegra dev 重新生成")
    print("      ren docker-compose.yml old.yml    # 改名留档")


section_2_pitfall()

### 预期输出

```text

==============================================================================
2. 课案点名的坑：目录里已有旧的 docker-compose.yml
==============================================================================
  课案原文：
      如果目录里已有旧的 docker-compose.yml（比如之前 langgraph 部署留下的），
      aegra dev 会直接用它而不是生成新的，先删掉或改名。

  为什么难查：aegra dev 不报错，用旧文件照样把容器起起来；
  但旧文件里的账号/口令/端口是上一套项目的 → 服务连不上库 →
  你看到的是「迁移失败 / 连接被拒绝」，而 .env 和 aegra.json 看起来都没问题。

  处理办法（课案给的两条任选其一）：
      del docker-compose.yml            # 删掉，让 aegra dev 重新生成
      ren docker-compose.yml old.yml    # 改名留档
```

> 上面两条处理办法是 Windows（cmd）写法。PowerShell 里对应
> `Remove-Item docker-compose.yml` 与 `Rename-Item docker-compose.yml old.yml`。

## 3. 生产部署：Dockerfile 逐行讲解

生产环境不用 `aegra dev`（那是开发用的），而是 `aegra up`：
构建镜像，然后把 **PostgreSQL + Redis + 应用**全部容器化起来。

课案的 Dockerfile 一共 10 行，每一行都有讲究：

```text
FROM python:3.12-slim

WORKDIR /app

# 国内 PyPI 镜像加速
ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 2026
CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]
```

In [ ]:
DOCKERFILE = '''FROM python:3.12-slim

WORKDIR /app

# 国内 PyPI 镜像加速
ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 2026
CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]
'''

# 逐行中文注释：键 = Dockerfile 里的原文行，值 = 这一行在干什么 / 为什么这么写
DOCKERFILE_ANNOTATIONS = [
    ("FROM python:3.12-slim",
     "基础镜像。slim 版去掉了编译工具链等用不到的东西，体积从 ~1GB 降到 ~150MB。"
     "选 3.12 是因为 Aegra 要求 Python 3.11+，而课案整个环境就是 3.12。"),
    ("WORKDIR /app",
     "容器内的工作目录。后面所有相对路径（COPY 的 . 、requirements.txt）都以它为准，"
     "CMD 里的 aegra 也在这个目录下执行 —— 所以 aegra.json 必须在 /app 根。"),
    ("ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple",
     "换清华 PyPI 镜像。**只在构建阶段生效**（ENV 在 RUN 时可见），"
     "国内服务器上不换源，装依赖那一步经常几十秒超时甚至失败。"),
    ("COPY requirements.txt .",
     "先单独 COPY 依赖清单，再 RUN 安装 —— 这是 Docker 分层缓存的经典技巧："
     "只要 requirements.txt 没变，改业务代码时这一层直接命中缓存，不用重装依赖。"),
    ("RUN pip install --no-cache-dir -r requirements.txt",
     "--no-cache-dir 不保留 pip 下载缓存，镜像更小（容器里不需要二次安装，缓存纯浪费）。"),
    ("COPY . .",
     "把项目代码复制进镜像。注意要配合 .dockerignore 用，"
     "否则本地 .env（真实密钥！）也会被打进镜像 —— 生产环境应当用环境变量注入。"),
    ("EXPOSE 2026",
     "声明容器监听 2026 端口。这行只是**文档性质**，不会真的开端口，"
     "真正对外暴露靠 docker-compose 的 ports 或 docker run -p。"),
    ('CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]',
     "用 exec 数组形式（不是 shell 字符串），保证 aegra 是 PID 1，"
     "能直接收到 docker stop 的 SIGTERM 优雅退出。"
     "--host 0.0.0.0 是必须的：绑 127.0.0.1 的话容器外访问不到。"
     "（回想第 2 节：aegra serve 在 Windows 上跑不了，但这行是在 Linux 容器里跑的，没问题。）"),
]

下面这一格：先把 Dockerfile 原样打出来，再逐行说明；最后给出服务器上的两条命令。

顺带把这份 Dockerfile **写到本课专属临时目录**（`WORKDIR / "aegra_deploy_client"`）——
源脚本是同目录下真放一个 `Dockerfile`，notebook 里没有「脚本同级目录」这个概念了，
所以统一落到临时目录，仓库里那份真实骨架（`09_aegra_deploy/aegra_project/`）只读不动。

In [ ]:
def section_3_production() -> None:
    print("\n" + "=" * 78)
    print("3. 生产部署：Dockerfile（逐行中文注释）")
    print("=" * 78)
    for ln in DOCKERFILE.rstrip("\n").splitlines():
        print(("    " + ln) if ln.strip() else "")

    print("\n  ◆ 逐行说明：")
    for code, note in DOCKERFILE_ANNOTATIONS:
        print(f"\n    {code}")
        # 说明较长，按显示宽度折行，窄终端也读得下去
        for seg in wrap_cjk(note, 66):
            print("        " + seg)

    print("\n  在服务器上（项目目录内）：")
    print_table(
        "生产部署命令",
        ["命令", "说明"],
        [
            ["aegra up", "构建镜像并启动 PostgreSQL + Redis + 应用"],
            ["aegra down", "停止（加 --volumes 会把数据卷一起删）"],
        ],
    )
    print("\n  所有服务自带健康检查和崩溃自动重启（compose 里的 healthcheck + restart）。")


section_3_production()

# 把这份 Dockerfile 落到本课专属临时目录（源脚本是放在项目根，notebook 里改到 WORKDIR 下）
NB_WORK = WORKDIR / "aegra_deploy_client"
NB_WORK.mkdir(exist_ok=True)
(NB_WORK / "Dockerfile").write_text(DOCKERFILE, encoding="utf-8")
print(f"\n  已把上面的 Dockerfile 写到临时目录：{NB_WORK / 'Dockerfile'}")

### 预期输出

```text

==============================================================================
3. 生产部署：Dockerfile（逐行中文注释）
==============================================================================
    FROM python:3.12-slim

    WORKDIR /app

    # 国内 PyPI 镜像加速
    ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple

    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt

    COPY . .

    EXPOSE 2026
    CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]

  ◆ 逐行说明：

    FROM python:3.12-slim
        基础镜像。slim 版去掉了编译工具链等用不到的东西，
        体积从 ~1GB 降到 ~150MB。选 3.12 是因为 Aegra 要求 Python 3.11+，
        而课案整个环境就是 3.12。

    WORKDIR /app
        容器内的工作目录。后面所有相对路径（COPY 的 . 、requirements.txt）
        都以它为准，CMD 里的 aegra 也在这个目录下执行 —— 所以 aegra.json 必
        须在 /app 根。

    ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple
        换清华 PyPI 镜像。**只在构建阶段生效**（ENV 在 RUN 时可见），
        国内服务器上不换源，装依赖那一步经常几十秒超时甚至失败。

    COPY requirements.txt .
        先单独 COPY 依赖清单，再 RUN 安装 —— 这是 Docker 分层缓存的经典技巧：
        只要 requirements.txt 没变，改业务代码时这一层直接命中缓存，
        不用重装依赖。

    RUN pip install --no-cache-dir -r requirements.txt
        --no-cache-dir 不保留 pip 下载缓存，
        镜像更小（容器里不需要二次安装，缓存纯浪费）。

    COPY . .
        把项目代码复制进镜像。注意要配合 .dockerignore 用，
        否则本地 .env（真实密钥！）也会被打进镜像 —— 生产环境应当用环境变量
        注入。

    EXPOSE 2026
        声明容器监听 2026 端口。这行只是**文档性质**，不会真的开端口，
        真正对外暴露靠 docker-compose 的 ports 或 docker run -p。

    CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]
        用 exec 数组形式（不是 shell 字符串），保证 aegra 是 PID 1，
        能直接收到 docker stop 的 SIGTERM 优雅退出。
        --host 0.0.0.0 是必须的：绑 127.0.0.1 的话容器外访问不到。
        （回想第 2 节：aegra serve 在 Windows 上跑不了，
        但这行是在 Linux 容器里跑的，没问题。）

  在服务器上（项目目录内）：

【生产部署命令】
+------------+------------------------------------------+
| 命令       | 说明                                     |
+------------+------------------------------------------+
| aegra up   | 构建镜像并启动 PostgreSQL + Redis + 应用 |
| aegra down | 停止（加 --volumes 会把数据卷一起删）    |
+------------+------------------------------------------+

  所有服务自带健康检查和崩溃自动重启（compose 里的 healthcheck + restart）。

  已把上面的 Dockerfile 写到临时目录：F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work\aegra_deploy_client\Dockerfile
```

三个最值得记的点：

1. **`COPY requirements.txt .` 单独一行** —— 分层缓存：requirements 不变时，
   改业务代码不会触发重装依赖（镜像构建从几分钟降到几秒）；
2. **`CMD` 用 exec 数组** —— `aegra` 成为 PID 1，能收到 `docker stop` 的 SIGTERM 优雅退出；
   写成 shell 字符串形式就只有 `/bin/sh` 收得到信号了；
3. **`--host 0.0.0.0`** —— 绑 `127.0.0.1` 的话容器外访问不到（这正是第 12 节那个
   「容器里的 localhost」话题的同源问题）。

## 4. 本机环境实测探测

上面三节都是「课案怎么说」。这一节真的去摸一遍**这台机器**：

| 探测项 | 为什么要探 |
|---|---|
| `docker` CLI | `aegra dev` / `up` 内部就是调 `docker compose` |
| docker **引擎** | CLI 装了 ≠ 引擎在跑 —— Windows 上必须开着 Docker Desktop |
| `aegra` 命令 | 本机没装属正常（要独立 3.12 环境） |
| `2026` 端口 | Aegra 默认监听端口，顺带判断服务是否已在跑 |
| 旧 `docker-compose.yml` | 第 2 节那个坑的**前置条件**在本机是否成立 |

> 「CLI 装了但引擎没跑」是最容易被忽略的一步：`docker --version` 正常，
> 一拉容器却报 `cannot connect to the Docker daemon`。

In [ ]:
# 源脚本里是 HERE = Path(__file__).resolve().parent + PROJECT_DIR = HERE / "aegra_project"
# —— notebook 里没有 __file__，改用 NB_DIR（notebook 所在目录，与源脚本所在目录相同）；
# 下面只是读它，全程只读，不写这个目录。
PROJECT_DIR = NB_DIR / "aegra_project"


def section_4_probe() -> None:
    """真的去查本机环境，把结果打印出来 —— 结论必须基于真实输出"""
    print("\n" + "=" * 78)
    print("4. 本机环境实测探测")
    print("=" * 78)

    # ---- 4.1 docker CLI ----
    print("\n  [4.1] Docker")
    docker_path = shutil.which("docker")
    if docker_path:
        ok, ver = run_cmd(["docker", "--version"])
        print(f"        CLI        ：{'已安装' if ok else '存在但调用失败'}  {docker_path}")
        print(f"        版本       ：{ver if ok else ver}")
        # docker CLI 装了 ≠ Docker 引擎在跑 —— Windows 上必须开着 Docker Desktop。
        # 这一步是最容易被忽略的：CLI 在，`docker --version` 也正常，但一拉容器就报
        # "error during connect / cannot connect to the Docker daemon"。
        ok2, srv = run_cmd(["docker", "info", "--format", "{{.ServerVersion}}"], timeout=15)
        if ok2:
            print(f"        引擎状态   ：✅ 运行中（Server {srv}）")
        else:
            print("        引擎状态   ：❌ 未运行 / 连不上")
            print("                     → Windows 上请先启动 Docker Desktop，等鲸鱼图标变绿再试")
        ok3, cver = run_cmd(["docker", "compose", "version"], timeout=15)
        print(f"        compose 插件：{'✅ ' + cver.splitlines()[0] if ok3 else '❌ ' + cver}")
        print("                     → aegra dev / up 内部就是调 docker compose，插件缺了跑不了")
    else:
        print("        ❌ 未找到 docker 命令")
        print("           → 装 Docker Desktop（Windows）后重开终端；aegra dev / up 都依赖它")
        print("           → 本节其余探测继续，不受影响")

    # ---- 4.2 aegra 命令 ----
    print("\n  [4.2] aegra 命令")
    aegra_path = shutil.which("aegra")
    if aegra_path:
        ok, ver = run_cmd(["aegra", "--version"], timeout=15)
        print(f"        已安装     ：{aegra_path}")
        print(f"        版本       ：{ver if ok else '（--version 不支持，可直接 aegra --help）'}")
    else:
        # 本机没装是正常情况：Aegra 要求独立的 3.12 环境，
        # 不能（也不该）装进 Python_Base 这个 venv —— 规范里明确禁止改依赖。
        print("        ❌ 未找到 aegra 命令（本机没装，属正常）")
        print("           → 按 02 节那三行在**独立环境**里装，不要装进 Python_Base 的 venv：")
        print("               conda create -n aegra python=3.12 -y")
        print("               conda activate aegra")
        print("               pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings")
        print("           → 本节其余内容（Dockerfile / 坑 / 流程）不受影响，纯讲解 + 探测")

    # ---- 4.3 2026 端口 ----
    print("\n  [4.3] 2026 端口（Aegra 默认监听）")
    # 用 socket.connect_ex 探测：比 ping / curl 快，也不依赖服务返回内容
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    in_use = sock.connect_ex(("127.0.0.1", 2026)) == 0
    sock.close()
    if in_use:
        print("        ✅ 127.0.0.1:2026 已被监听 —— 可能 Aegra 已经在跑了")
        print("           → 直接试 curl.exe http://localhost:2026/health")
    else:
        print("        ⚪ 127.0.0.1:2026 没有服务在监听")
        print("           → 说明本地还没起 Aegra；启动方式见本文件第 1 节（aegra dev）")

    # ---- 4.4 课案那个坑，在本机的真实情况 ----
    print("\n  [4.4] 课案那个坑在本机的真实情况")
    # 02 节刚刚生成过 docker-compose.yml，所以这里**必然**能演示出这个坑的前置条件
    compose = PROJECT_DIR / "docker-compose.yml"
    if compose.exists():
        size = compose.stat().st_size
        print(f"        ⚠️ {compose}")
        print(f"           已存在（{size} 字节，02 节生成的）")
        print("           → 现在直接跑 aegra dev，它会**沿用这份**而不是重新生成。")
        print("           → 这正是课案提醒的场景：确认这份 compose 就是你想要的那份；")
        print("             如果它是别的项目留下的，先 del 或 ren 掉再跑 aegra dev。")
    else:
        print(f"        ⚪ {PROJECT_DIR} 下没有 docker-compose.yml")
        print("           → 首次运行 aegra dev 会自动生成一份，不会触发这个坑")


section_4_probe()

# 源 03 主流程的最后两行（保留，方便与源脚本输出对照）
print("\n小结：本地 aegra dev 一条命令搞定（注意旧 compose 文件的坑）；")
print("      生产 aegra up 走 Dockerfile + compose 全容器化（aegra serve 只在 Linux 上跑）。")

### 预期输出

```text

==============================================================================
4. 本机环境实测探测
==============================================================================

  [4.1] Docker
        CLI        ：已安装  C:\Program Files\Docker\Docker\resources\bin\docker.EXE
        版本       ：Docker version 29.7.2, build a7dcaa6
        引擎状态   ：✅ 运行中（Server 29.7.2）
        compose 插件：✅ Docker Compose version v5.5.0
                     → aegra dev / up 内部就是调 docker compose，插件缺了跑不了

  [4.2] aegra 命令
        ❌ 未找到 aegra 命令（本机没装，属正常）
           → 按 02 节那三行在**独立环境**里装，不要装进 Python_Base 的 venv：
               conda create -n aegra python=3.12 -y
               conda activate aegra
               pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings
           → 本节其余内容（Dockerfile / 坑 / 流程）不受影响，纯讲解 + 探测

  [4.3] 2026 端口（Aegra 默认监听）
        ⚪ 127.0.0.1:2026 没有服务在监听
           → 说明本地还没起 Aegra；启动方式见本文件第 1 节（aegra dev）

  [4.4] 课案那个坑在本机的真实情况
        ⚠️ F:\ProGram\Python_Base\Agent\09_aegra_deploy\aegra_project\docker-compose.yml
           已存在（2362 字节，02 节生成的）
           → 现在直接跑 aegra dev，它会**沿用这份**而不是重新生成。
           → 这正是课案提醒的场景：确认这份 compose 就是你想要的那份；
             如果它是别的项目留下的，先 del 或 ren 掉再跑 aegra dev。
```

```text
小结：本地 aegra dev 一条命令搞定（注意旧 compose 文件的坑）；
      生产 aegra up 走 Dockerfile + compose 全容器化（aegra serve 只在 Linux 上跑）。
```

**本机结论**（与输出逐条对应）：

| 探测项 | 本机结果 | 说明 |
|---|---|---|
| docker CLI | ✅ 已安装 | `aegra dev` / `up` 的基础 |
| docker 引擎 | ✅ 运行中 | Docker Desktop 起着，容器能拉 |
| compose 插件 | ✅ 有 | `aegra dev` 内部调的就是它 |
| `aegra` 命令 | ❌ 未安装 | 预期状态：要独立 3.12 环境，不装进本项目 venv |
| `2026` 端口 | ⚪ 没服务 | 本地没起 Aegra → 第 9 节会自动切离线演示 |
| 旧 compose 文件 | ⚠️ 已存在 | 第 2 节那个坑的**前置条件在本机真的成立** |

## 第二部分 · 客户端调用（源 `04_client`）

服务起来之后（或者起不来，走离线），客户端怎么调？
这一篇的核心只有两句：`client.threads.create()` 和 `client.runs.stream(...)`。

最关键的认知是：**这段客户端代码在 Aegra 和官方 LangSmith Deployments 之间完全通用，
只换 `url`** —— 这正是上一课「客户端 SDK 同款」那一格的实际含义。

## 5. 课案原文：客户端调用（约 30 行）

```text
from langgraph_sdk import get_sync_client
client = get_sync_client(url="http://localhost:2026")

thread = client.threads.create()          # 创建会话线程（状态持久化在 PostgreSQL）
for chunk in client.runs.stream(
    thread["thread_id"],
    "bushu",                              # 对应 aegra.json 中 graphs 的 key
    input={"messages": [{"type": "human", "content": "你好"}]},
    stream_mode="messages",
):
    ...
```

下面把这段原文**整段存成字符串**打出来对照（后面用的都是据此重构出来的版本）。
先记住两个约定：

1. 每个 graph 启动时会自动注册一个**同名默认 assistant**，所以 `assistant_id`
   直接填 graph 名即可；同一 `thread_id` 连续调用就是**多轮对话**；
2. `stream_mode="messages"` 的事件名是 `messages/partial`（**带斜杠**）——
   因为它要作为 SSE 的 `event:` 字段发给前端。

In [ ]:
COURSE_SNIPPET = '''import sys

from langgraph_sdk import get_sync_client

# Windows 控制台默认 GBK 编码，遇到 emoji 会抛 UnicodeEncodeError，强制 UTF-8
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

client = get_sync_client(url="http://localhost:2026")

# 创建会话线程（状态持久化在 PostgreSQL）
thread = client.threads.create()

# 流式调用
printed = 0  # 已打印的字符数
for chunk in client.runs.stream(
    thread["thread_id"],
    "bushu",      # 对应 aegra.json 中 graphs 的 key
    input={
        "messages": [{"type": "human", "content": "你好"}]
    },
    stream_mode="messages",
):
    # messages/partial 事件的 data 是 [消息块, 元数据]
    # 注意：content 是“累计全文”而非增量 token，需自己计算差量打印
    if chunk.event == "messages/partial":
        content = chunk.data[0]["content"]
        if isinstance(content, str) and len(content) > printed:
            print(content[printed:], end="", flush=True)
            printed = len(content)
'''


def section_1_snippet() -> None:
    print("=" * 78)
    print("1. 课案原文：客户端调用（约 30 行）")
    print("=" * 78)
    for ln in COURSE_SNIPPET.rstrip("\n").splitlines():
        print("    " + ln)
    print("\n  说明：每个 graph 启动时会自动注册一个**同名默认 assistant**，")
    print("        所以 assistant_id 直接填 graph 名即可。")
    print("        同一 thread_id 连续调用就是多轮对话。")


print("Agent 课案 · 部署 ④：客户端调用（langgraph_sdk + messages/partial 差量打印）")
section_1_snippet()

### 预期输出

```text
Agent 课案 · 部署 ④：客户端调用（langgraph_sdk + messages/partial 差量打印）
==============================================================================
1. 课案原文：客户端调用（约 30 行）
==============================================================================
    import sys

    from langgraph_sdk import get_sync_client

    # Windows 控制台默认 GBK 编码，遇到 emoji 会抛 UnicodeEncodeError，强制 UTF-8
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

    client = get_sync_client(url="http://localhost:2026")

    # 创建会话线程（状态持久化在 PostgreSQL）
    thread = client.threads.create()

    # 流式调用
    printed = 0  # 已打印的字符数
    for chunk in client.runs.stream(
        thread["thread_id"],
        "bushu",      # 对应 aegra.json 中 graphs 的 key
        input={
            "messages": [{"type": "human", "content": "你好"}]
        },
        stream_mode="messages",
    ):
        # messages/partial 事件的 data 是 [消息块, 元数据]
        # 注意：content 是“累计全文”而非增量 token，需自己计算差量打印
        if chunk.event == "messages/partial":
            content = chunk.data[0]["content"]
            if isinstance(content, str) and len(content) > printed:
                print(content[printed:], end="", flush=True)
                printed = len(content)

  说明：每个 graph 启动时会自动注册一个**同名默认 assistant**，
        所以 assistant_id 直接填 graph 名即可。
        同一 thread_id 连续调用就是多轮对话。
```

注意课案原文里那两个细节：`printed = 0`（**自己记已打印字符数**）与
`content[printed:]`（**只打新增的那一段**）。第 7 节会把「为什么必须这样」讲透。

## 6. `assistant_id` 从哪来：核对真实 `aegra.json`

`assistant_id` **必须**和 `aegra.json` 里 `graphs` 的 key 一致，否则请求会 **404**。

课案两处写法不一致：**「调用」那节写 `bushu`、「项目结构」那节写 `agent`** ——
这是课案作者的笔误。下面直接读上一课生成的**真实 `aegra.json`** 来核对，
不一致就自动改用真实注册的那个 key。

In [ ]:
def registered_graph_keys() -> list:
    """读 aegra.json，返回 graphs 里注册的所有 key（= assistant_id 候选）

    读不到就返回空列表 —— 这是探测，不抛异常。
    """
    cfg = PROJECT_DIR / "aegra.json"
    try:
        data = json.loads(cfg.read_text(encoding="utf-8"))
        return list(data.get("graphs", {}).keys())
    except Exception:   # noqa: BLE001 —— 文件不存在/JSON 坏了都当成「没读到」
        return []


def section_2_assistant_id() -> str:
    """打印 assistant_id 与 aegra.json 的对应关系，返回本次实际使用的 id"""
    print("\n" + "=" * 78)
    print("2. assistant_id 从哪来：核对 aegra.json 的 graphs key")
    print("=" * 78)
    keys = registered_graph_keys()
    if keys:
        print(f"  读到 {PROJECT_DIR / 'aegra.json'}")
        print(f"  已注册的 graph（= assistant_id）：{keys}")
    else:
        print(f"  ⚪ 没读到 {PROJECT_DIR / 'aegra.json'}（先跑 02_项目骨架_jxsd.py 生成）")

    if keys and COURSE_ASSISTANT_ID in keys:
        print(f"  ✅ 课案用的 \"{COURSE_ASSISTANT_ID}\" 在已注册列表里，可直接调用")
        return COURSE_ASSISTANT_ID
    if keys:
        print(f"  ⚠️ 课案写的 \"{COURSE_ASSISTANT_ID}\" **不在**已注册列表里 → 真调用会 404。")
        print(f"     课案「调用」写 bushu、「项目结构」写 agent，两处不一致是课案的笔误；")
        print(f"     解决办法二选一：改 aegra.json 的 key，或改客户端的 assistant_id。")
        print(f"     本次演示自动改用已注册的 \"{keys[0]}\"。")
        return keys[0]
    print(f"  ⚪ 没有可用列表，本次演示按课案原值 \"{COURSE_ASSISTANT_ID}\" 走（离线演示不受影响）")
    return COURSE_ASSISTANT_ID

课案里那个 `"bushu"` 是个常量，先在下面声明（**必须**和 `aegra.json` 的 `graphs` key 一致）：

In [ ]:
# 和课案一致：Aegra 默认监听 2026
BASE_URL = "http://localhost:2026"
HEALTH_URL = f"{BASE_URL}/health"

# 课案「调用」那段里写的 assistant_id 是 "bushu" ——
# 注意它**必须**和 aegra.json 里 graphs 的 key 一致，否则请求会 404。
# 课案作者的 aegra.json 里 key 就叫 bushu；而课案「项目结构」那节给的示例
# 是 "agent"。下面 register_target() 会去读真实生成的 aegra.json 帮你核对。
COURSE_ASSISTANT_ID = "bushu"

In [ ]:
assistant_id = section_2_assistant_id()

### 预期输出

```text

==============================================================================
2. assistant_id 从哪来：核对 aegra.json 的 graphs key
==============================================================================
  读到 F:\ProGram\Python_Base\Agent\09_aegra_deploy\aegra_project\aegra.json
  已注册的 graph（= assistant_id）：['agent']
  ⚠️ 课案写的 "bushu" **不在**已注册列表里 → 真调用会 404。
     课案「调用」写 bushu、「项目结构」写 agent，两处不一致是课案的笔误；
     解决办法二选一：改 aegra.json 的 key，或改客户端的 assistant_id。
     本次演示自动改用已注册的 "agent"。
```

**本机真实情况**：仓库里那份 `aegra_project/aegra.json` 注册的 key 是 `agent`，
而课案客户端写的是 `bushu` —— **不在列表里**，所以走了 ⚠️ 分支并自动改用 `agent`。

这就是「课案笔误」在现场长什么样：代码没问题、服务也没问题，
但客户端一调就 404，而且报错信息不会告诉你「是 key 写错了」。
排查时先回来看 `aegra.json`，比读服务端日志快。

## 7. 核心：流式事件的数据结构

这一节的绝对重点。`stream_mode="messages"` 的事件有两个反直觉之处：

| 反直觉点 | 事实 | 后果 |
|---|---|---|
| 事件名 | 是 `messages/partial` / `messages/complete`（**带斜杠**） | 前端按 SSE 的 `event:` 字段分发，所以名字里不能有空格 |
| `chunk.data` | 是一个**两元素列表** `[消息块, 元数据]` | `data[0]` 才是内容；`data[1]` 告诉你「来自哪个节点」 |
| `content` | 是**累计全文**，不是增量 token | 想做打字机效果必须自己记 `printed` 再取差量 |

```text
data[0] = {"content": ..., "type": "AIMessageChunk", ...}      真正的内容
data[1] = {"langgraph_node": "chatbot", "tags": [...], ...}    来自哪个节点
```

下面这段 `consume_stream()` **就是课案那 30 行里的循环体，一个字都没改** ——
只是把 `client.runs.stream(...)` 的结果作为参数传进来，
这样在线模式（真 SSE）和离线模式（假 chunk）能跑**同一份逻辑**。

In [ ]:
def consume_stream(chunks, show_events: bool = False) -> None:
    """消费 `stream_mode="messages"` 的事件流，做出打字机效果

    这一段就是课案那 30 行里的循环体，**一个字都没改**，只是把
    `client.runs.stream(...)` 的结果作为参数传进来 ——
    这样在线模式（真 SSE）和离线模式（假 chunk）能跑同一份逻辑。

    show_events=True 时不打打字机，改成「一个事件一行」，把累计全文和差量并排显示，
    方便看清「content 是累计的」这件事。
    """
    printed = 0        # 已打印的字符数（课案原句：printed = 0  # 已打印的字符数）
    partial_count = 0  # 顺带统计事件数，方便观察「累计全文」是怎么长的

    for chunk in chunks:
        # messages/partial 事件的 data 是 [消息块, 元数据]
        # 注意：content 是“累计全文”而非增量 token，需自己计算差量打印
        if chunk.event == "messages/partial":
            content = chunk.data[0]["content"]
            if isinstance(content, str) and len(content) > printed:
                if show_events:
                    # 把「累计全文」和「算出来的差量」并排打出来，一眼看出为什么要 [printed:]
                    print(f"      [event] {chunk.event:<18} 累计={content!r}   →  本次只打 {content[printed:]!r}")
                else:
                    print(content[printed:], end="", flush=True)
                printed = len(content)
            partial_count += 1
        elif show_events:
            # metadata / messages/complete 等事件没有 data[0]["content"]，
            # 靠这个 else 直观说明：循环里那个 if chunk.event == 判断不是可选的。
            print(f"      [event] {chunk.event:<18} （非 partial，跳过：没有 data[0]['content']）")

    print()   # 流结束换行（打字机效果期间一直没换行）
    print(f"      ↑ 共收到 {partial_count} 个 messages/partial 事件，最终全文 {printed} 个字符")

## 8. 离线降级：手工构造假 chunk

本机现在没起 Aegra（第 4 节探测过），所以第 9 节的真实调用分支走不到。
但**看不到 SSE 事件**不等于学不到东西 —— 这里手工把事件造出来，
用同一段 `consume_stream()` 跑一遍。

⭐ `build_fake_chunks()` 是本节最该盯着看的地方：每次事件的
`data[0]["content"]` 都是**从头累积**的字符串，而不是那一个 token。

In [ ]:
class FakeChunk:
    """模拟 langgraph_sdk 的 StreamPart：只有 .event 和 .data 两个属性被用到

    真实类型是 langgraph_sdk.schema.StreamPart（NamedTuple），字段一致：
        event: str   事件名，例如 "messages/partial"
        data:  Any   事件负载，messages 模式下是 [消息块, 元数据]
    """

    def __init__(self, event: str, data):
        self.event = event
        self.data = data


def build_fake_chunks() -> list:
    """把一个回答拆成 token，再拼成「累计全文」的 messages/partial 事件序列

    ⭐ 这里是本节最值得盯着看的地方：每一次事件的 data[0]["content"] 都是
       **从头累积**的字符串，而不是那一个 token。所以课案才要
       `content[printed:]` 自己算差量。
    """
    tokens = ["你", "好", "！", "我是", "部署", "在", "Aegra", "上", "的", "助手", "。"]
    meta = {"langgraph_node": "chatbot", "tags": [], "ls_provider": "openai"}

    chunks = [
        # 真实调用时，stream 的第一个事件通常是 metadata（run 的基本信息），
        # 它没有 data[0]["content"] —— 这就是循环里必须写 `if chunk.event ==` 的原因
        FakeChunk("metadata", {"run_id": "1ef7c1a2-fake-run-id", "attempt": 1}),
    ]
    cumulative = ""
    for tok in tokens:
        cumulative += tok
        chunks.append(
            FakeChunk(
                "messages/partial",
                # data 是两元素列表：[消息块, 元数据]
                [{"content": cumulative, "type": "AIMessageChunk", "id": "lc_run--fake"}, meta],
            )
        )
    chunks.append(
        FakeChunk(
            "messages/complete",
            [{"content": cumulative, "type": "AIMessage", "id": "lc_run--fake"}, meta],
        )
    )
    return chunks

`FakeClient` 把**接口形状**也对齐真客户端（`client.threads` / `client.runs`），
这样离线模式下的调用方式与在线完全一致，学员不用在两套代码之间来回换算。

In [ ]:
class FakeClient:
    """假的 langgraph_sdk 客户端，接口形状和真的对齐：client.threads / client.runs

    这样 consume_stream 里的调用方式（threads.create() → runs.stream(...)）
    在离线模式下也**完全一致**，学员不用在两套代码之间来回换算。
    """

    class _Threads:
        @staticmethod
        def create():
            return {"thread_id": "fake-thread-0001", "created_at": "2025-01-01T00:00:00Z"}

    class _Runs:
        @staticmethod
        def stream(thread_id, assistant_id, input, stream_mode):
            # 真客户端这里是发 HTTP 请求、逐条 yield SSE 事件；
            # 离线模式直接返回预构造好的事件列表（同样是可迭代的）
            return build_fake_chunks()

    def __init__(self):
        self.threads = self._Threads()
        self.runs = self._Runs()

离线演示跑两遍：第一遍是**打字机效果**（和课案那 30 行一样），
第二遍加 `show_events=True`，把「累计」和「差量」**并排**打出来 ——
那一眼就能看出为什么必须写 `content[printed:]`。

In [ ]:
def section_4_offline(assistant_id: str) -> None:
    """离线演示：用假 chunk 把课案那段差量打印逻辑真跑一遍"""
    print("\n" + "=" * 78)
    print("4. 离线演示：手工构造 messages/partial 事件，跑一遍课案那段逻辑")
    print("=" * 78)

    chunks = build_fake_chunks()
    print(f"  构造了 {len(chunks)} 个事件。开头三个的原样长这样：")
    for c in chunks[:3]:
        print(f"\n    event = {c.event!r}")
        print("    data  = " + json.dumps(c.data, ensure_ascii=False)[:150] + " ...")
    print("\n  ⚠️ 看后两个：第一个 partial 的 content 是「你」，第二个是「你好」——")
    print("     它每次都是**从开头累积**的完整字符串，不是那一个 token。")
    print("     所以课案才要 `content[printed:]` 自己算差量。")

    print("\n  ---- 下面是课案那段循环体的真实输出（打字机效果）----")
    print("      ", end="")
    client = FakeClient()
    thread = client.threads.create()          # 和真客户端同形
    consume_stream(
        client.runs.stream(
            thread["thread_id"],
            assistant_id,                      # 对应 aegra.json 中 graphs 的 key
            input={"messages": [{"type": "human", "content": "你好"}]},
            stream_mode="messages",
        ),
        show_events=False,
    )

    print("\n  ---- 再跑一次，这次一个事件一行，把「累计」和「差量」并排看 ----")
    consume_stream(FakeClient().runs.stream(
        "fake-thread-0001", assistant_id,
        input={"messages": [{"type": "human", "content": "你好"}]},
        stream_mode="messages",
    ), show_events=True)

## 9. 探测 + 主流程：在线或离线

两个探测器：

- `health_ok()`：`httpx.get(HEALTH_URL, timeout=2.0)`，任何异常都算「没起」；
- `section_3_health()`：打印探测结果，不通时给出**完整的启动步骤**
  （`cp .env.example .env` → 确认 Docker → 确认没有旧 compose → `uv run aegra dev`）。

`section_5_online()` 是真实调用路径，和离线演示**共用**同一个 `consume_stream()`。
注意它把整个调用包在 `try/except` 里：服务端报错不该把课案脚本炸掉。

In [ ]:
def health_ok() -> bool:
    """探测 Aegra 是否在跑（2 秒超时，任何异常都算「没起」）"""
    try:
        import httpx   # langgraph_sdk 的依赖，一定装了
        resp = httpx.get(HEALTH_URL, timeout=2.0)
        return resp.status_code == 200
    except Exception:   # noqa: BLE001 —— 连接失败/超时/包缺失都归为「连不上」
        return False


def section_3_health() -> bool:
    print("\n" + "=" * 78)
    print("3. 探测本机 Aegra 服务")
    print("=" * 78)
    print(f"  GET {HEALTH_URL}   （httpx，2 秒超时）")
    ok = health_ok()
    if ok:
        print('  ✅ 通！返回 {"status": "healthy"} 之类，服务已就绪')
    else:
        print("  ❌ 连不上 —— 本机现在没有 Aegra 在 2026 端口上跑")
        print("\n  想把它跑起来（完整步骤见 03_本地开发与生产部署_jxsd.py）：")
        print(f"      1) cd {PROJECT_DIR}")
        print("      2) cp .env.example .env      # 然后填上真实的 API_KEY 等（.env 不进 Git）")
        print("      3) 确认 Docker Desktop 已经启动（aegra dev 要拉 PostgreSQL 容器）")
        print("      4) 确认目录里没有别的项目留下的旧 docker-compose.yml（课案点名的坑）")
        print("      5) uv run aegra dev          # 自动：生成 compose → 拉 PostgreSQL → 迁移 → 热重载")
        print("      6) 另开一个终端：curl.exe http://localhost:2026/health")
        print("      7) 交互式文档：http://localhost:2026/docs")
        print("\n  注意 aegra 要装在独立的 3.12 环境里，不要装进 Python_Base 的 venv。")
    return ok

In [ ]:
def section_5_online(assistant_id: str) -> None:
    """服务在跑时的真实调用 —— 与离线演示共用 consume_stream"""
    print("\n" + "=" * 78)
    print("5. 在线调用（真实 langgraph_sdk）")
    print("=" * 78)
    try:
        from langgraph_sdk import get_sync_client

        client = get_sync_client(url=BASE_URL)
        thread = client.threads.create()
        print(f"  已创建线程：{thread['thread_id']}")
        print(f"  调用 assistant_id = {assistant_id}，stream_mode = messages")
        print("  AI：", end="")
        consume_stream(
            client.runs.stream(
                thread["thread_id"],
                assistant_id,
                input={"messages": [{"type": "human", "content": "你好"}]},
                stream_mode="messages",
            ),
        )
        # 同一个 thread_id 再问一次，就是多轮对话（上下文由服务端 PostgreSQL 里的
        # checkpoint 维持，客户端不需要把历史一起发过去 —— 这正是 Threads API 的价值）
        print("\n  同一 thread 再问一次（验证多轮上下文）：")
        print("  AI：", end="")
        consume_stream(
            client.runs.stream(
                thread["thread_id"],
                assistant_id,
                input={"messages": [{"type": "human", "content": "我叫什么？"}]},
                stream_mode="messages",
            ),
        )
    except Exception as exc:   # noqa: BLE001 —— 服务端报错不该把课案脚本炸掉
        print(f"  ❌ 调用失败：{type(exc).__name__}: {exc}")
        print("     常见原因：assistant_id 和 aegra.json 的 key 不一致（会 404）；")
        print("               或服务刚起来还在跑数据库迁移，等几秒再试。")

主流程就是源脚本 `if __name__ == "__main__":` 里那段 `if / else`（顶格写出）：
**探测得通就走在线，不通就走离线演示**。

In [ ]:
if section_3_health():
    section_5_online(assistant_id)
else:
    section_4_offline(assistant_id)
    print("\n" + "=" * 78)
    print("小结")
    print("=" * 78)
    print("  · 客户端代码在 Aegra 和官方平台之间**完全通用**，只换 url；")
    print("  · messages/partial 的 content 是累计全文，要自己 [printed:] 算差量；")
    print("  · 事件名带斜杠（messages/partial），因为要作为 SSE 的 event 字段发给前端；")
    print("  · 循环里那个 `if chunk.event ==` 不是可选的 —— metadata 等事件没有 content。")
    print("\n  服务起来后再跑一次本文件，就会自动切到第 4 节的真实调用路径。")

### 预期输出

```text

==============================================================================
3. 探测本机 Aegra 服务
==============================================================================
  GET http://localhost:2026/health   （httpx，2 秒超时）
  ❌ 连不上 —— 本机现在没有 Aegra 在 2026 端口上跑

  想把它跑起来（完整步骤见 03_本地开发与生产部署_jxsd.py）：
      1) cd F:\ProGram\Python_Base\Agent\09_aegra_deploy\aegra_project
      2) cp .env.example .env      # 然后填上真实的 API_KEY 等（.env 不进 Git）
      3) 确认 Docker Desktop 已经启动（aegra dev 要拉 PostgreSQL 容器）
      4) 确认目录里没有别的项目留下的旧 docker-compose.yml（课案点名的坑）
      5) uv run aegra dev          # 自动：生成 compose → 拉 PostgreSQL → 迁移 → 热重载
      6) 另开一个终端：curl.exe http://localhost:2026/health
      7) 交互式文档：http://localhost:2026/docs

  注意 aegra 要装在独立的 3.12 环境里，不要装进 Python_Base 的 venv。

==============================================================================
4. 离线演示：手工构造 messages/partial 事件，跑一遍课案那段逻辑
==============================================================================
  构造了 13 个事件。开头三个的原样长这样：

    event = 'metadata'
    data  = {"run_id": "1ef7c1a2-fake-run-id", "attempt": 1} ...

    event = 'messages/partial'
    data  = [{"content": "你", "type": "AIMessageChunk", "id": "lc_run--fake"}, {"langgraph_node": "chatbot", "tags": [], "ls_provider": "openai"}] ...

    event = 'messages/partial'
    data  = [{"content": "你好", "type": "AIMessageChunk", "id": "lc_run--fake"}, {"langgraph_node": "chatbot", "tags": [], "ls_provider": "openai"}] ...

  ⚠️ 看后两个：第一个 partial 的 content 是「你」，第二个是「你好」——
     它每次都是**从开头累积**的完整字符串，不是那一个 token。
     所以课案才要 `content[printed:]` 自己算差量。

  ---- 下面是课案那段循环体的真实输出（打字机效果）----
      你好！我是部署在Aegra上的助手。
      ↑ 共收到 11 个 messages/partial 事件，最终全文 18 个字符

  ---- 再跑一次，这次一个事件一行，把「累计」和「差量」并排看 ----
      [event] metadata           （非 partial，跳过：没有 data[0]['content']）
      [event] messages/partial   累计='你'   →  本次只打 '你'
      [event] messages/partial   累计='你好'   →  本次只打 '好'
      [event] messages/partial   累计='你好！'   →  本次只打 '！'
      [event] messages/partial   累计='你好！我是'   →  本次只打 '我是'
      [event] messages/partial   累计='你好！我是部署'   →  本次只打 '部署'
      [event] messages/partial   累计='你好！我是部署在'   →  本次只打 '在'
      [event] messages/partial   累计='你好！我是部署在Aegra'   →  本次只打 'Aegra'
      [event] messages/partial   累计='你好！我是部署在Aegra上'   →  本次只打 '上'
      [event] messages/partial   累计='你好！我是部署在Aegra上的'   →  本次只打 '的'
      [event] messages/partial   累计='你好！我是部署在Aegra上的助手'   →  本次只打 '助手'
      [event] messages/partial   累计='你好！我是部署在Aegra上的助手。'   →  本次只打 '。'
      [event] messages/complete  （非 partial，跳过：没有 data[0]['content']）

      ↑ 共收到 11 个 messages/partial 事件，最终全文 18 个字符

==============================================================================
小结
==============================================================================
  · 客户端代码在 Aegra 和官方平台之间**完全通用**，只换 url；
  · messages/partial 的 content 是累计全文，要自己 [printed:] 算差量；
  · 事件名带斜杠（messages/partial），因为要作为 SSE 的 event 字段发给前端；
  · 循环里那个 `if chunk.event ==` 不是可选的 —— metadata 等事件没有 content。

  服务起来后再跑一次本文件，就会自动切到第 4 节的真实调用路径。
```

三个值得停一下看的细节：

1. `第 3 节` 打印 `❌ 连不上` 之后给出 7 步启动清单 —— 这是**设计好的降级**，不是失败；
2. 打字机那一段打出的是完整回答（`你好！我是部署在 Aegra 上的助手。`），
   而它是靠 `content[printed:]` 一次一个 token 拼出来的；
3. 第二遍 `show_events=True` 里，每个事件的 `累计=` 都在变长，而 `本次只打=` 只有那一小段
   —— **这就是「累计全文 vs 增量」最直观的证据**。

> 另外注意 `metadata` 那一行打的是「非 partial，跳过」：
> 它没有 `data[0]["content"]`，所以循环里那个 `if chunk.event == ...` **不是可选的**。

## 第三部分 · 对接 Langfuse（源 `05_langfuse`）

Agent 上生产之后必须能回答三个问题：**它为什么这么答？花了多少 token？慢在哪一步？**
这就是可观测（observability）。Aegra 的追踪走 **OpenTelemetry + Langfuse**。

课案给了**两条接入路径**，水平和代价都不一样，别混为一谈：

| | 路径 A | 路径 B |
|---|---|---|
| 改什么 | `.env` 四行 | `graph.py` 里挂 `CallbackHandler` |
| 视角 | **服务端视角**（一次 run / 一次 graph step） | **LangChain 内部视角**（每条链 / 每次 LLM / 每个工具） |
| 原理 | Aegra 内置 OTEL 导出器，读到 `OTEL_TARGETS=LANGFUSE` 就推 | LangChain 喊事件 → handler 翻译成 OTel span |
| 代价 | 零代码 | 多几行代码、key 要自己传 |

两条路可以**同时开**，在 Langfuse 上会看到粗细两级的 span 拼成同一棵 trace。

## 10. 路径 A：`.env` 里加四行，代码零改动

课案原文（推荐路径）：

```text
OTEL_TARGETS="LANGFUSE"
LANGFUSE_BASE_URL="http://localhost:3000"   # 自托管 Langfuse 地址
LANGFUSE_PUBLIC_KEY="pk-lf-xxx"
LANGFUSE_SECRET_KEY="sk-lf-xxx"
```

In [ ]:
ENV_LINES = [
    'OTEL_TARGETS="LANGFUSE"',
    'LANGFUSE_BASE_URL="http://localhost:3000"   # 自托管 Langfuse 地址',
    'LANGFUSE_PUBLIC_KEY="pk-lf-xxx"',
    'LANGFUSE_SECRET_KEY="sk-lf-xxx"',
]


def section_1_env_only() -> None:
    print("=" * 78)
    print("1. 路径 A：.env 里加四行，代码零改动")
    print("=" * 78)
    for ln in ENV_LINES:
        print("    " + ln)
    print("\n  OTEL_TARGETS 是「开关」：Aegra 服务读到 LANGFUSE 就启用 Langfuse 导出器；")
    print("  剩下三个是「地址 + 钥匙」，缺一个都推不上去。")
    print_table(
        "四个变量的作用",
        ["变量", "作用"],
        [
            ["OTEL_TARGETS", "选择 OTEL 导出目标；值 LANGFUSE 即开启 Langfuse 追踪"],
            ["LANGFUSE_BASE_URL", "Langfuse 服务地址（自托管就填自己的，如 http://localhost:3000）"],
            ["LANGFUSE_PUBLIC_KEY", "公钥，pk-lf- 开头，Langfuse 项目设置里生成"],
            ["LANGFUSE_SECRET_KEY", "私钥，sk-lf- 开头 —— **只在本地 .env，不要提交进仓库**"],
        ],
    )
    print("\n  ⚠️ 密钥只写在本地 .env 里，不要提交进仓库（.env 已在 .gitignore 中）。")
    print("  ⚠️ 容器里的 localhost 是容器自己 —— 见下面第 3 节那个 host.docker.internal 的坑。")


print("Agent 课案 · 部署 ⑤：对接 Langfuse（.env 零代码 + CallbackHandler）")
section_1_env_only()

### 预期输出

```text
Agent 课案 · 部署 ⑤：对接 Langfuse（.env 零代码 + CallbackHandler）
==============================================================================
1. 路径 A：.env 里加四行，代码零改动
==============================================================================
    OTEL_TARGETS="LANGFUSE"
    LANGFUSE_BASE_URL="http://localhost:3000"   # 自托管 Langfuse 地址
    LANGFUSE_PUBLIC_KEY="pk-lf-xxx"
    LANGFUSE_SECRET_KEY="sk-lf-xxx"

  OTEL_TARGETS 是「开关」：Aegra 服务读到 LANGFUSE 就启用 Langfuse 导出器；
  剩下三个是「地址 + 钥匙」，缺一个都推不上去。

【四个变量的作用】
+---------------------+-----------------------------------------------------------------+
| 变量                | 作用                                                            |
+---------------------+-----------------------------------------------------------------+
| OTEL_TARGETS        | 选择 OTEL 导出目标；值 LANGFUSE 即开启 Langfuse 追踪            |
| LANGFUSE_BASE_URL   | Langfuse 服务地址（自托管就填自己的，如 http://localhost:3000） |
| LANGFUSE_PUBLIC_KEY | 公钥，pk-lf- 开头，Langfuse 项目设置里生成                      |
| LANGFUSE_SECRET_KEY | 私钥，sk-lf- 开头 —— **只在本地 .env，不要提交进仓库**          |
+---------------------+-----------------------------------------------------------------+

  ⚠️ 密钥只写在本地 .env 里，不要提交进仓库（.env 已在 .gitignore 中）。
  ⚠️ 容器里的 localhost 是容器自己 —— 见下面第 3 节那个 host.docker.internal 的坑。
```

四个变量分两类：**`OTEL_TARGETS` 是开关**，另外三个是「地址 + 两把钥匙」。
服务的改动为零 —— 改完 `.env` 重启服务（`aegra dev` / `aegra up`）即生效。

## 11. 路径 B：`graph.py` 里手工挂 `CallbackHandler`

课案原文（`graph.py`），注意课案写的是 `from config import setting` / `setting.XXX`，
本课按项目配置规范**统一改成 `settings` / `settings.xxx`**：

```text
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

langfuse = Langfuse(public_key=..., secret_key=..., host=...)
langfuse_handler = CallbackHandler()

graph = create_deep_agent(model=model, tools=[], config={"callbacks": [langfuse_handler]})
```

In [ ]:
GRAPH_PY_LANGFUSE = '''from langfuse import Langfuse
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

from config import settings

langfuse = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    host=settings.langfuse_host,
)
langfuse_handler = CallbackHandler()

model = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

graph = create_deep_agent(model=model, tools=[], config={"callbacks": [langfuse_handler]})
'''

CALLBACK_PRINCIPLE = [
    ("LangChain 在执行时会「喊事件」",
     "每进入一个 Runnable 就喊 on_chain_start，调用模型喊 on_chat_model_start、"
     "模型返回喊 on_llm_end，工具调用喊 on_tool_start / on_tool_end …… "
     "这些喊话是 LangChain 内建的机制，不需要你写任何埋点代码。"),
    ("CallbackHandler 就是「听事件的人」",
     "它继承自 langchain_core.callbacks.BaseCallbackHandler，把 on_xxx 方法实现一遍即可。"
     "Langfuse 的 CallbackHandler 做的事就是：每听到一个事件，就开/关一个 OTel span，"
     "把输入输出、token 数、耗时挂上去，再批量发给 Langfuse。"),
    ("怎么让 handler 听到：config={\"callbacks\": [...]}",
     "把 handler 放进 config 里，LangChain 会在**整棵调用树**上传播它 —— "
     "所以 create_deep_agent(...) 里挂一次，规划、子 Agent、工具调用全都会被记录，"
     "不用逐个节点挂。这是它比手工埋点省事的地方。"),
    ("它的信号源和路径 A 不一样",
     "路径 A 记的是「服务端视角」（一次 run / 一次 graph step）；"
     "路径 B 记的是「LangChain 内部视角」（每一条链、每一次 LLM、每一个工具）。"
     "两条路可以同时开，在 Langfuse 上会看到粗细两级的 span 拼成同一棵 trace。"),
]


def section_2_graph_py() -> None:
    print("\n" + "=" * 78)
    print("2. 路径 B：graph.py 里手工挂 CallbackHandler（逐行讲解）")
    print("=" * 78)
    for ln in GRAPH_PY_LANGFUSE.rstrip("\n").splitlines():
        print("    " + ln)

    print("\n  ◆ CallbackHandler 的原理（四步）")
    for i, (title, detail) in enumerate(CALLBACK_PRINCIPLE, 1):
        print(f"\n    {i}. {title}")
        text = detail
        while text:
            print("       " + text[:68])
            text = text[68:]


section_2_graph_py()

### 预期输出

```text

==============================================================================
2. 路径 B：graph.py 里手工挂 CallbackHandler（逐行讲解）
==============================================================================
    from langfuse import Langfuse
    from langfuse.langchain import CallbackHandler
    from langchain_openai import ChatOpenAI
    from deepagents import create_deep_agent

    from config import settings

    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,
    )
    langfuse_handler = CallbackHandler()

    model = ChatOpenAI(
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )

    graph = create_deep_agent(model=model, tools=[], config={"callbacks": [langfuse_handler]})

  ◆ CallbackHandler 的原理（四步）

    1. LangChain 在执行时会「喊事件」
       每进入一个 Runnable 就喊 on_chain_start，调用模型喊 on_chat_model_start、模型返回喊 on_
       llm_end，工具调用喊 on_tool_start / on_tool_end …… 这些喊话是 LangChain 内建的机制，不
       需要你写任何埋点代码。

    2. CallbackHandler 就是「听事件的人」
       它继承自 langchain_core.callbacks.BaseCallbackHandler，把 on_xxx 方法实现一遍即可。
       Langfuse 的 CallbackHandler 做的事就是：每听到一个事件，就开/关一个 OTel span，把输入输出、toke
       n 数、耗时挂上去，再批量发给 Langfuse。

    3. 怎么让 handler 听到：config={"callbacks": [...]}
       把 handler 放进 config 里，LangChain 会在**整棵调用树**上传播它 —— 所以 create_deep_ag
       ent(...) 里挂一次，规划、子 Agent、工具调用全都会被记录，不用逐个节点挂。这是它比手工埋点省事的地方。

    4. 它的信号源和路径 A 不一样
       路径 A 记的是「服务端视角」（一次 run / 一次 graph step）；路径 B 记的是「LangChain 内部视角」（每一条
       链、每一次 LLM、每一个工具）。两条路可以同时开，在 Langfuse 上会看到粗细两级的 span 拼成同一棵 trace。
```

第 4 条最容易被忽略：**两条路的信号源不同**，不是「A 的替代品」。
路径 A 看到的是 `run` / `graph step` 这一级；路径 B 看到的是链 / LLM / 工具那一级。

> 注意课案原文里的 `create_deep_agent(model=model, tools=[], ...)`：
> 这里只是为了演示「handler 挂在哪」；真按 Aegra 骨架写，
> 图应该是 `aegra_project/my_agent/graph.py` 里那个。

## 12. 容器里的 `localhost` 陷阱

课案 `docker-compose.yml` 里专门写了注释：

```yaml
environment:
  ...
  # 启用 Langfuse 追踪；容器内 localhost 指向自身，需走宿主机网关访问 langfuse
  - OTEL_TARGETS=LANGFUSE
  - LANGFUSE_BASE_URL=http://host.docker.internal:3000
```

原因：**Langfuse 跑在宿主机，Agent 跑在容器里；容器内的 `localhost` 是容器自己**。
填 `http://localhost:3000` 的话，容器会去找自己身上根本不存在的 Langfuse，
表现是「配置看起来都对，但 Langfuse 上一条数据都没有」——
这是自托管 Langfuse + 容器化 Agent 最常见的「配了但收不到数据」的原因。

还有一个优先级细节：**`.env` 里的 `LANGFUSE_BASE_URL` 会被 compose 的 `environment` 覆盖**
（同一个键，`environment` 优先级高于 `env_file`）。所以两边可以各写各的：
本地 `.env` 填 `localhost`（宿主机直接跑时用），compose 里写
`host.docker.internal`（容器里跑时用），互不干扰。

In [ ]:
def section_3_docker_localhost() -> None:
    print("\n" + "=" * 78)
    print("3. 容器里的 localhost 陷阱（课案 docker-compose 的注释）")
    print("=" * 78)
    print("  课案 docker-compose.yml 里这几行：")
    print("      environment:")
    print("        # 启用 Langfuse 追踪；容器内 localhost 指向自身，需走宿主机网关访问 langfuse")
    print("        - OTEL_TARGETS=LANGFUSE")
    print("        - LANGFUSE_BASE_URL=http://host.docker.internal:3000")
    print("\n  为什么：Langfuse 跑在宿主机，Agent 跑在容器里。")
    print("          容器内的 localhost = **容器自己**，不是宿主机 ——")
    print("          填 http://localhost:3000 的话，容器会去找自己身上根本不存在的 Langfuse，")
    print("          表现是「配置看起来都对，但 Langfuse 上一条数据都没有」。")
    print("          host.docker.internal 是 Docker Desktop 给的宿主机别名，走这个才通。")
    print("\n  另外注意 .env 里的 LANGFUSE_BASE_URL 会被 compose 的 environment **覆盖**：")
    print("      同一个键，environment 优先级高于 env_file。")
    print("      所以本地 .env 填 localhost（宿主机直接跑时用），")
    print("      compose 里再改成 host.docker.internal（容器里跑时用），两边互不干扰。")


section_3_docker_localhost()

### 预期输出

```text

==============================================================================
3. 容器里的 localhost 陷阱（课案 docker-compose 的注释）
==============================================================================
  课案 docker-compose.yml 里这几行：
      environment:
        # 启用 Langfuse 追踪；容器内 localhost 指向自身，需走宿主机网关访问 langfuse
        - OTEL_TARGETS=LANGFUSE
        - LANGFUSE_BASE_URL=http://host.docker.internal:3000

  为什么：Langfuse 跑在宿主机，Agent 跑在容器里。
          容器内的 localhost = **容器自己**，不是宿主机 ——
          填 http://localhost:3000 的话，容器会去找自己身上根本不存在的 Langfuse，
          表现是「配置看起来都对，但 Langfuse 上一条数据都没有」。
          host.docker.internal 是 Docker Desktop 给的宿主机别名，走这个才通。

  另外注意 .env 里的 LANGFUSE_BASE_URL 会被 compose 的 environment **覆盖**：
      同一个键，environment 优先级高于 env_file。
      所以本地 .env 填 localhost（宿主机直接跑时用），
      compose 里再改成 host.docker.internal（容器里跑时用），两边互不干扰。
```

本机这份 `.env` 里 `langfuse_host` 是 `http://localhost:3001`（第 15 节会看到）——
那正是因为**它就是宿主机直接跑的那个进程**，还没有进容器。
一旦 Agent 进了容器，这个值就必须换成 `host.docker.internal`。

> 顺带一提：第 3 节 Dockerfile 里 `--host 0.0.0.0` 的讲究，和这里是同一类问题
> —— `127.0.0.1` 永远只代表「我自己」。

## 13. 课案 `requirements.txt` 那 8 个包，各自干什么

```text
aegra-cli
aegra-api
langgraph
langchain
langchain-openai
deepagents
langfuse
pydantic-settings
```

In [ ]:
REQUIREMENTS = [
    "aegra-cli",
    "aegra-api",
    "langgraph",
    "langchain",
    "langchain-openai",
    "deepagents",
    "langfuse",
    "pydantic-settings",
]

REQUIREMENTS_DOC = {
    "aegra-cli": "命令行工具：aegra init / dev / up / down / serve。只在你敲命令时需要（生产镜像里其实可以不要，但留着方便进容器排查）",
    "aegra-api": "服务本体：用 FastAPI 实现的那套 Agent Protocol（threads / runs / assistants / store）",
    "langgraph": "图引擎（MIT 开源）：StateGraph、checkpoint、streaming 都来自它 —— 部署平台收费，它不收费",
    "langchain": "LangChain 主包：Runnable / 回调 / 工具那套公共抽象，也是 langchain-core 的统一入口",
    "langchain-openai": "模型接入层：ChatOpenAI，走 OpenAI 兼容协议（DeepSeek / grok / 通义 都靠它接）",
    "deepagents": "create_deep_agent：课案 graph.py 用它把「规划 + 子 Agent + 工具」组装成一张现成的图",
    "langfuse": "可观测平台 SDK：提供 Langfuse 客户端与 langfuse.langchain.CallbackHandler",
    "pydantic-settings": "从 .env 读配置：课案的 conf.py 就是靠它的 BaseSettings（本项目用根目录 config.py）",
}


def section_4_requirements() -> None:
    print("\n" + "=" * 78)
    print("4. 课案 requirements.txt 那 8 个包，各自干什么")
    print("=" * 78)
    for name in REQUIREMENTS:
        print(f"    {name}")
    print_table(
        "8 个依赖的分工",
        ["包名", "作用"],
        [[name, REQUIREMENTS_DOC[name]] for name in REQUIREMENTS],
    )
    print("\n  一句话：前 2 个是 Aegra 自己的（命令行 + 服务），")
    print("          中间 4 个是你写 Agent 用的，langfuse 是观测，pydantic-settings 是配置。")


section_4_requirements()

### 预期输出

```text

==============================================================================
4. 课案 requirements.txt 那 8 个包，各自干什么
==============================================================================
    aegra-cli
    aegra-api
    langgraph
    langchain
    langchain-openai
    deepagents
    langfuse
    pydantic-settings

【8 个依赖的分工】
+-------------------+----------------------------------------------------------------------------------------------------------------------+
| 包名              | 作用                                                                                                                 |
+-------------------+----------------------------------------------------------------------------------------------------------------------+
| aegra-cli         | 命令行工具：aegra init / dev / up / down / serve。只在你敲命令时需要（生产镜像里其实可以不要，但留着方便进容器排查） |
| aegra-api         | 服务本体：用 FastAPI 实现的那套 Agent Protocol（threads / runs / assistants / store）                                |
| langgraph         | 图引擎（MIT 开源）：StateGraph、checkpoint、streaming 都来自它 —— 部署平台收费，它不收费                             |
| langchain         | LangChain 主包：Runnable / 回调 / 工具那套公共抽象，也是 langchain-core 的统一入口                                   |
| langchain-openai  | 模型接入层：ChatOpenAI，走 OpenAI 兼容协议（DeepSeek / grok / 通义 都靠它接）                                        |
| deepagents        | create_deep_agent：课案 graph.py 用它把「规划 + 子 Agent + 工具」组装成一张现成的图                                  |
| langfuse          | 可观测平台 SDK：提供 Langfuse 客户端与 langfuse.langchain.CallbackHandler                                            |
| pydantic-settings | 从 .env 读配置：课案的 conf.py 就是靠它的 BaseSettings（本项目用根目录 config.py）                                   |
+-------------------+----------------------------------------------------------------------------------------------------------------------+

  一句话：前 2 个是 Aegra 自己的（命令行 + 服务），
          中间 4 个是你写 Agent 用的，langfuse 是观测，pydantic-settings 是配置。
```

一句话记法：**前 2 个是 Aegra 自己的**（命令行 + 服务），
**中间 4 个是写 Agent 用的**（`langgraph` / `langchain` / `langchain-openai` / `deepagents`），
**`langfuse` 是观测，`pydantic-settings` 是配置**。

## 14. 降级方案：一个「打印型」`CallbackHandler`

本机 Langfuse 密钥是**齐的**，所以第 15 节会走真实 handler。
但这一格仍然要放在这里 —— 因为它是理解 Langfuse 的**入口**：

| | Langfuse 的 handler | 这个「打印型」handler |
|---|---|---|
| 基类 | `BaseCallbackHandler` | `BaseCallbackHandler` |
| 接收方式 | 重写 `on_xxx` | 重写 `on_xxx` |
| 听到之后 | 开/关 OTel span，批量发给 Langfuse | `print` 出来，什么也不发 |

它们的**接口完全相同**，差别只在「听到事件之后做什么」。所以没有密钥的环境里，
用它来演示「Langfuse 上那条 trace 是由哪些事件拼出来的」是完全等价的替身。

实现上的两个要点：

- `run_id` / `parent_run_id` 是 LangChain 给每次执行分配的 ID，
  靠 `parent` 指针就能还原出**调用树的层次**（下面用缩进表示）；
- `on_llm_new_token` **只在流式调用时才触发**，`invoke` 不会 ——
  保留它是为了说明「流式和回调是两套机制」。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import BaseCallbackHandler


class PrintingCallbackHandler(BaseCallbackHandler):
    """把 LangChain 的回调事件直接打印到屏幕上 —— 用来看清「事件长什么样」

    它和 langfuse.langchain.CallbackHandler **是同一个接口**：
    都继承 BaseCallbackHandler，都靠重写 on_xxx 方法来接收事件。
    区别只在于「听到之后做什么」：

        Langfuse 的 handler → 开/关 OTel span，批量发给 Langfuse 服务
        这个 handler        → print 出来，什么也不发

    所以在没有 Langfuse 密钥的环境里，用它来演示「Langfuse 上那条 trace
    是由哪些事件拼出来的」，是完全等价的教学替身。

    实现要点：
      · run_id / parent_run_id 是 LangChain 给每次执行分配的 ID，
        靠 parent 指针就能还原出调用树的层次（下面用缩进表示）；
      · on_llm_new_token 只在**流式**调用时才触发，invoke 不会触发 ——
        这里保留它是为了说明「流式和回调是两套机制」。
    """

    def __init__(self) -> None:
        super().__init__()
        self.events = []        # (事件名, 详情) 供结尾汇总
        self._parents = {}      # run_id -> parent_run_id，用来算缩进层级

    def _record(self, kind: str, run_id, name: str, detail: str = "", parent_run_id=None) -> None:
        """记录并打印一条事件，按父子关系缩进"""
        self._parents[run_id] = parent_run_id
        depth, cur = 0, parent_run_id
        while cur is not None and depth < 10:
            depth += 1
            cur = self._parents.get(cur)
        # 打上 run_id 前 8 位：**同一次执行的 start/end 用的是同一个 run_id**，
        # 这正是 Langfuse 把两个事件合成一个 span 的依据。
        line = f"        {'    ' * depth}└─ [{kind}] {name}  (run={str(run_id)[:8]})"
        if detail:
            line += f"  {detail}"
        print(line)
        self.events.append((kind, detail or name))

    # ---------- 链（Runnable）事件 ----------
    def on_chain_start(self, serialized, inputs, *, run_id, parent_run_id=None, **kwargs):
        name = (serialized or {}).get("name") or "Runnable"
        self._record("chain_start", run_id, name, f"输入键={list((inputs or {}).keys())}",
                     parent_run_id)

    def on_chain_end(self, outputs, *, run_id, parent_run_id=None, **kwargs):
        keys = list(outputs.keys()) if isinstance(outputs, dict) else type(outputs).__name__
        self._record("chain_end", run_id, "链结束", f"输出={keys}", parent_run_id)

    # ---------- 大模型事件 ----------
    def on_chat_model_start(self, serialized, messages, *, run_id, parent_run_id=None, **kwargs):
        name = (serialized or {}).get("name") or "ChatModel"
        n = len(messages[0]) if messages else 0
        self._record("chat_model_start", run_id, name, f"发送 {n} 条消息", parent_run_id)

    def on_llm_new_token(self, token, *, run_id, parent_run_id=None, **kwargs):
        # invoke（非流式）不会触发这个事件 —— 只有 stream 才会
        self._record("llm_new_token", run_id, "新 token", repr(token), parent_run_id)

    def on_llm_end(self, response, *, run_id, parent_run_id=None, **kwargs):
        detail = ""
        try:
            gen = response.generations[0][0]
            text = getattr(getattr(gen, "message", None), "content", None) or getattr(gen, "text", "")
            usage = (response.llm_output or {}).get("token_usage") or {}
            if usage:
                # 只挑三个最关键的字段，整份 usage 里还塞着一堆 None，全打出来没法看
                total = usage.get("total_tokens")
                reason = (usage.get("completion_tokens_details") or {}).get("reasoning_tokens")
                detail = f"total_tokens={total}"
                if reason:
                    detail += f"（其中推理 {reason}）"
                detail += "  "
            detail += f"回答前 20 字={text[:20]!r}"
        except Exception:   # noqa: BLE001 —— 回调里绝不能再抛异常
            detail = "（结构解析失败，跳过）"
        self._record("llm_end", run_id, "模型返回", detail, parent_run_id)

    def on_llm_error(self, error, *, run_id, parent_run_id=None, **kwargs):
        self._record("llm_error", run_id, "模型报错", f"{type(error).__name__}", parent_run_id)

    # ---------- 工具事件 ----------
    def on_tool_start(self, serialized, input_str, *, run_id, parent_run_id=None, **kwargs):
        name = (serialized or {}).get("name") or "Tool"
        self._record("tool_start", run_id, name, f"参数={input_str[:40]!r}", parent_run_id)

    def on_tool_end(self, output, *, run_id, parent_run_id=None, **kwargs):
        self._record("tool_end", run_id, "工具返回", f"{str(output)[:40]!r}", parent_run_id)

    def on_tool_error(self, error, *, run_id, parent_run_id=None, **kwargs):
        self._record("tool_error", run_id, "工具报错", f"{type(error).__name__}", parent_run_id)

## 15. 本机 Langfuse 配置检查 + 真实追踪

最后一节把两条路**真跑一遍**：读配置 → 密钥齐全就用**真实**
`langfuse.langchain.CallbackHandler`，否则降级为上面的「打印型」handler →
然后**真的调一次大模型**，把回调事件（或真实上传结果）打出来。

> ⚠️ 源码这一节里有一处**实测踩坑**，已经修好，这里必须照搬：
> 只有「打印型」handler 才有 `.events` 属性；真实的
> `langfuse.langchain.CallbackHandler` **没有**这个属性
> （它把事件直接发去服务端，本地不留）。无条件取会 `AttributeError`，
> 所以结尾用 `getattr(handler, "events", None)` 判空。

In [ ]:
def section_5_keys_and_demo() -> None:
    """读 Langfuse 配置：配好了走真 handler，没配走降级演示"""
    print("\n" + "=" * 78)
    print("5. 本机 Langfuse 配置检查 + 回调事件演示")
    print("=" * 78)

    public_key = settings.langfuse_public_key
    secret_key = settings.langfuse_secret_key
    host = settings.langfuse_host
    print(f"  settings.langfuse_public_key : {public_key!r}")
    print(f"  settings.langfuse_secret_key : {'（已设置）' if secret_key else repr(secret_key)}")
    print(f"  settings.langfuse_host       : {host!r}")

    handler = None
    real_langfuse = False
    if public_key and secret_key:
        # 密钥齐全 → 用课案那套真实 CallbackHandler
        try:
            from langfuse import Langfuse
            from langfuse.langchain import CallbackHandler

            Langfuse(public_key=public_key, secret_key=secret_key, host=host)
            handler = CallbackHandler()
            real_langfuse = True
            print("\n  ✅ 密钥齐全，使用真实的 langfuse.langchain.CallbackHandler")
        except Exception as exc:   # noqa: BLE001 —— 包缺了/版本不对都降级
            print(f"\n  ⚠️ 初始化 Langfuse 失败（{type(exc).__name__}: {exc}），降级为打印型 handler")

    if handler is None:
        # 密钥为空（本仓库的实际情况）→ 打印中文提示 + 降级。
        # 降级而不是直接退出，是因为本节要教的核心其实是「Langfuse 收到的是什么」，
        # 而这件事用打印型 handler 就能完整演示 —— 没有密钥不该让这一节白跑。
        print("\n  ❌ Langfuse 密钥为空 —— 追踪链路走不通，本次走降级演示（不抛异常）")
        print("\n  怎么配上（两条路任选，细节见本文件第 1 / 2 节）：")
        print("     路径 A（零代码，推荐）：在 aegra_project/.env 里加四行")
        for ln in ENV_LINES:
            print("         " + ln)
        print("         → 然后重启服务（aegra dev / aegra up），不用改任何 .py")
        print("     路径 B（代码级）：把第 2 节那段 graph.py 里的 CallbackHandler 挂到 agent 上")
        print("\n     自托管 Langfuse 的启动（在 Langfuse 自己的仓库目录里）：")
        print("         docker compose up -d      # 默认就是 3000 端口")
        print("     然后在 Langfuse 控制台 → Settings → API Keys 生成 pk-lf- / sk-lf- 一对，")
        print("     填进本地 .env 的 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY。")
        print("\n     ⚠️ 密钥只写在本地 .env，不要提交进仓库、也不要贴进聊天记录。")
        print("\n  ---- 降级：用一个「打印型」CallbackHandler，真的调一次大模型 ----")
        print("      它和 Langfuse 的 handler 接口完全相同，只是把事件 print 出来而不是发出去。\n")
        handler = PrintingCallbackHandler()

    # ---------- 真实调用大模型（这一步会真的发请求）----------
    llm = init_chat_model(
        model_provider="openai",
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )
    question = "用一句话说明 Langfuse 能观测到 Agent 的哪些东西"
    print(f"  提问：{question}")
    print(f"  模型：{settings.model_name}（{settings.base_url}）\n")
    print("  ---- 回调事件序列（缩进 = 调用树的层次）----")
    try:
        response = llm.invoke(question, config={"callbacks": [handler]})
        print("\n  ---- 事件结束 ----")
        answer = getattr(response, "content", str(response))
        print(f"\n  AI：{answer}")
        print("\n  ◆ 怎么读上面这串事件：")
        print("    · chat_model_start 和 llm_end 的 run= 前缀是**同一个** ——")
        print("      它们描述的是同一次执行的开始与结束，Langfuse 靠 run_id 把两个事件")
        print("      合成一个 generation span（耗时 = 两者时间差，输入输出各挂一头）。")
        print("    · 这里有工具调用的话，会多出 tool_start / tool_end 两层，并用缩进体现父子关系。")
        print("    · ⚠️ 直接 llm.invoke 只会触发模型级事件，**看不到 chain_start/chain_end**；")
        print("      把模型包进链（RunnableSequence）或 Agent 之后，chain 那一层才出现 ——")
        print("      课案 graph.py 里 create_deep_agent 挂 handler，记的就是这一整棵树。")
        if real_langfuse:
            # Langfuse v4：flush() 立即上传，否则要等批量刷新，控制台里可能看不到
            from langfuse import get_client
            get_client().flush()
            print(f"\n  已上传到 Langfuse：{host} → 去控制台看这条 trace")
        else:
            print("\n  （以上事件就是 Langfuse 那条 trace 的原料：")
            print("    chat_model_start → llm_end 这一段会被记成一个 generation span，")
            print("    token 数 / 耗时 / 输入输出都会挂在上面。）")
    except Exception as exc:   # noqa: BLE001 —— 网络/额度问题不该把演示炸掉
        print(f"\n  ❌ 调用模型失败：{type(exc).__name__}: {exc}")
        print("     检查 .env 里的 API_KEY / BASE_URL / MODEL_NAME 是否可用。")
        return

    # 结尾汇总：只有「打印型」handler 会把事件同时存进 `.events`；真实的
    # `langfuse.langchain.CallbackHandler` **没有**这个属性（它把事件直接发去服务端，
    # 本地不留），无条件取会 `AttributeError` —— 实测踩坑，故用 getattr 判空。
    # 事件明细在上面已经边收边打印过了，这里只报个数 + 说明事件与 span 的对应关系。
    events = getattr(handler, "events", None)
    if events is None:
        print("\n  （真实 Langfuse handler 不在本地留事件，去控制台看这条 trace 的 span 树。）")
    else:
        print(f"\n  本次共捕获 {len(events)} 个回调事件")
        print("    · chat_model_start / llm_end 成对出现 → 合成一个 generation span")
        print("    · chain_start / chain_end             → 合成一个 chain span（父节点）")
        print("    · tool_start / tool_end               → 合成一个 tool span")


section_5_keys_and_demo()

# 源 05 主流程的最后两行（保留，方便与源脚本输出对照）
print("\n小结：追踪有两条路 —— .env 四行（服务端视角，零代码）与")
print("      graph.py 挂 CallbackHandler（LangChain 内部视角）；")
print("      容器里访问宿主机的 Langfuse 记得用 host.docker.internal。")

### 预期输出

```text

==============================================================================
5. 本机 Langfuse 配置检查 + 回调事件演示
==============================================================================
  settings.langfuse_public_key : 'pk-lf-…（已配置；真实值按约定不写进输出块）'
  settings.langfuse_secret_key : （已设置）
  settings.langfuse_host       : 'http://localhost:3001'

  ✅ 密钥齐全，使用真实的 langfuse.langchain.CallbackHandler
  提问：用一句话说明 Langfuse 能观测到 Agent 的哪些东西
  模型：deepseek-flash（https://api.deepseek.com）

  ---- 回调事件序列（缩进 = 调用树的层次）----

  ---- 事件结束 ----

  AI：（模型回答，每次都不一样；本次是）
      Langfuse 能追踪 Agent 从请求到响应的完整执行链，包括每一步的嵌套步骤、工具调用、
      检索与大模型生成、Prompt 与输入输出、延迟、Token/成本、错误异常，
      以及用户/会话元数据和评估分数。

  ◆ 怎么读上面这串事件：
    · chat_model_start 和 llm_end 的 run= 前缀是**同一个** ——
      它们描述的是同一次执行的开始与结束，Langfuse 靠 run_id 把两个事件
      合成一个 generation span（耗时 = 两者时间差，输入输出各挂一头）。
    · 这里有工具调用的话，会多出 tool_start / tool_end 两层，并用缩进体现父子关系。
    · ⚠️ 直接 llm.invoke 只会触发模型级事件，**看不到 chain_start/chain_end**；
      把模型包进链（RunnableSequence）或 Agent 之后，chain 那一层才出现 ——
      课案 graph.py 里 create_deep_agent 挂 handler，记的就是这一整棵树。

  已上传到 Langfuse：http://localhost:3001 → 去控制台看这条 trace

  （真实 Langfuse handler 不在本地留事件，去控制台看这条 trace 的 span 树。）
```

```text

小结：追踪有两条路 —— .env 四行（服务端视角，零代码）与
      graph.py 挂 CallbackHandler（LangChain 内部视角）；
      容器里访问宿主机的 Langfuse 记得用 host.docker.internal。
```

**本机走的是真实追踪路径**（密钥齐全），所以：

1. 打印 `✅ 密钥齐全，使用真实的 langfuse.langchain.CallbackHandler`；
2. 「回调事件序列」那一段**中间没有事件行** —— 真实 handler 把事件直接发去服务端，
   本地不留，这正是源码那句注释说的事；如果走了降级路径，这里会刷出一串
   `└─ [chat_model_start] ...`；
3. 结尾打的是「真实 Langfuse handler 不在本地留事件」——
   **如果无条件取 `handler.events`，这里就会 `AttributeError`**；
4. 最后 `已上传到 Langfuse：http://localhost:3001` 之后，去控制台就能看到这条 trace，
   上面挂着一个 `generation` span（模型、token 数、耗时、输入输出都在上面）。

> 公钥那一行本 notebook **打码**了：源脚本会原样打印 `.env` 里的真实值，
> 而这份「预期输出」是要进仓库的 —— 按本项目约定，任何输出块都不落地密钥明文。

## 小结

这一课把「部署 → 调用 → 观测」串成了一条链：

| 环节 | 一句话 |
|---|---|
| 本地开发 | `uv run aegra dev` = 生成 compose → 拉 PostgreSQL → 跑迁移 → 热重载起服务 |
| 那个坑 | 目录里已有旧 `docker-compose.yml` 时 `aegra dev` **直接沿用**，不报错、只表现为连不上库 |
| 生产部署 | `aegra up` 走 Dockerfile + compose 全容器化；`CMD` 用 exec 数组让 aegra 当 PID 1 |
| 客户端 | `threads.create()` + `runs.stream(...)`；**换 url 就能在 Aegra 和官方平台之间切换** |
| 流式结构 | 事件名带斜杠（`messages/partial`）；`data` 是 `[消息块, 元数据]`；`content` 是**累计全文** |
| 追踪 · 路径 A | `.env` 四行 `OTEL_TARGETS=LANGFUSE` + 地址 + 两把钥匙，代码零改动 |
| 追踪 · 路径 B | `graph.py` 里 `config={"callbacks": [CallbackHandler()]}`，挂一次覆盖整棵树 |
| 可观测的原理 | LangChain 喊事件 → handler 翻译成 OTel span → trace 上一棵 span 树 |

**下一课怎么衔接**：`10_workflow_platform` 会跳到另一个工作流平台；
而这一章的三件套（骨架 → 跑起来 / 调用 → 观测）已经构成了一个**能交付的最小部署方案**。

## 常见坑

1. **`aegra dev` 不报错但库连不上** → 先看目录里有没有**旧的 `docker-compose.yml`**
   （第 2 节）。`.env` 和 `aegra.json` 都是对的，问题不在这儿。
2. **`assistant_id` 与 `aegra.json` 的 key 不一致会 404** ——
   课案自己就写混了（`bushu` vs `agent`）。先读 `aegra.json`，别猜。
3. **`messages/partial` 的 `content` 是累计全文**，不是增量 token。
   直接 `print(content)` 会把整段回答重复打印 N 遍；要 `content[printed:]`。
4. **事件名在代码里是带斜杠的字符串**（`"messages/partial"`），
   别写成 `"partial"` —— 它对应 SSE 的 `event:` 字段。
5. **循环里那个 `if chunk.event == ...` 不是可选的**：
   `metadata` / `messages/complete` 等事件没有 `data[0]["content"]`，
   不加判断会 `KeyError` / `IndexError`。
6. **容器里访问宿主机的 Langfuse 必须用 `host.docker.internal`**，
   写 `localhost` 是「配了但收不到数据」的第一大原因（第 12 节）。
7. **`.env` 里的 `LANGFUSE_BASE_URL` 会被 compose 的 `environment` 覆盖**
   —— 同一个键，`environment` 优先级更高。两边写不同值是有意为之，不是冲突。
8. **真实 `langfuse.langchain.CallbackHandler` 没有 `.events` 属性**（实测）：
   只有自己写的「打印型」handler 才有。取事件列表要用 `getattr(handler, "events", None)`，
   否则 `AttributeError`。
9. **`aegra serve` 在 Windows 上跑不了** —— 它只跑在 Linux 容器里（Dockerfile 那行 `CMD`）。
   想在 Windows 本地跑服务，用 `aegra dev`（它走 compose，不是原生）。
10. **PowerShell 里 `curl` 是 `Invoke-WebRequest` 的别名**，参数不通用；
    写 `curl.exe` 或改用 `Invoke-RestMethod`。

## 官方链接

- Aegra 仓库（Apache 2.0，LangGraph Platform 的自托管替代）：<https://github.com/aegra/aegra>
- Aegra 文档（`aegra init / dev / up` 命令与 `aegra.json`）：<https://github.com/aegra/aegra#readme>
- LangGraph SDK（客户端 `threads` / `runs` 接口）：<https://docs.langchain.com/langsmith/sdk>
- LangGraph 流式（`stream_mode="messages"` 的事件结构）：<https://docs.langchain.com/oss/python/langgraph/streaming>
- LangChain 回调系统（`BaseCallbackHandler` 的 `on_xxx` 方法表）：<https://docs.langchain.com/oss/python/langchain/callbacks>
- Langfuse × LangChain（`CallbackHandler` 用法）：<https://langfuse.com/integrations/frameworks/langchain>
- Langfuse 自托管（`docker compose up -d` 与 API Keys）：<https://langfuse.com/self-hosting>
- OpenTelemetry（Aegra 追踪的底层协议）：<https://opentelemetry.io/docs/>
- Dockerfile 参考（`CMD` exec 形式 / `EXPOSE` / 分层缓存）：<https://docs.docker.com/reference/dockerfile/>